# Evaluation 3 – Manual check: answers, pipeline, KQML messages and SQL

This notebook goes through **all 100 questions one by one (Q1 → Q100)** and shows, for each question,
everything you need to judge the answer yourself:

| step | cell | what you see |
|---|---|---|
| 1 | `show(...)`, `answer(...)`, `diff(...)` | every distinct answer of the 40 asks (4 models × 2 agents × 5 reps) and who gave it |
| 2 | `pipeline(...)` | per model and agent: the category the LLM chose, whether extraction was right, SQL steps / attempts / errors, KQML turns, time and tokens |
| 3 | `kqml(...)` | the KQML messages the agent sent (`ask`) and got back (`tell`), **explained in plain words** |
| 4 | SQL check | a query that computes the true answer from both databases, **explained above the cell**; `compare(...)` matches it with each answer |
| 5 | `review(...)` | your verdict per answer (`y` / `n` / `p`) and your written review, saved to `review/answer_review.csv` |

Run sections 0–2 once, read the short guide in section 3, then work through the questions.
The verdicts go into the same file as notebook 02, so you can use either notebook.

Note: the result files do not store the SQL text the agents generated – only how many SQL steps and attempts they
needed and the SQL error messages (shown by `pipeline`). The SQL in this notebook is the **check** query.

## 0. Setup

In [39]:
import json
import os
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

DATA_DIR = Path("..") / "data"
REVIEW_DIR = Path("..") / "review"
REVIEW_FILE = REVIEW_DIR / "answer_review.csv"
PER_ASK_FILE = REVIEW_DIR / "answer_verdicts_per_ask.csv"
REVIEW_DIR.mkdir(exist_ok=True)

MODELS = ["gpt-4o-mini", "claude-haiku", "qwen3-8b", "qwen3-8b-lora"]
MODEL_LABEL = {
    "gpt-4o-mini": "GPT-4o mini",
    "claude-haiku": "Claude Haiku 5.5",
    "qwen3-8b": "Qwen3-8B",
    "qwen3-8b-lora": "Qwen3-8B + QLoRA",
}
VERDICTS = {"y": "correct", "n": "wrong", "p": "partly",
            "correct": "correct", "wrong": "wrong", "partly": "partly"}
REVIEW_COLUMNS = ["qid", "question", "category", "fingerprint", "asks", "given_by",
                  "answer_status", "answer_short", "verdict", "sql", "sql_result", "note", "reviewed_at"]

In [40]:
def read_env(path):
    """Reads KEY=VALUE lines from a .env file (no extra package needed)."""
    values = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                values[key.strip()] = value.split(" #")[0].strip().strip('"').strip("'")
    return values


ENV_FILE = Path("..") / ".env"
env = {**read_env(ENV_FILE), **{k: v for k, v in os.environ.items() if k.startswith("DB_")}}


def db_settings(agent_no):
    get = lambda key, default=None: env.get(f"{key}_AGENT{agent_no}", env.get(key, default))
    return {"host": get("DB_HOST", "localhost"), "port": get("DB_PORT", "5432"), "user": get("DB_USER", "postgres"),
            "password": get("DB_PASS", ""), "dbname": get("DB_NAME", f"agent_{agent_no}_db")}


DB_URLS = {"Agent-001": db_settings(1), "Agent-002": db_settings(2)}
SQL_MAX_ROWS = 5000

print(f".env found: {ENV_FILE.resolve()}" if ENV_FILE.exists() else f"No .env at {ENV_FILE.resolve()} – using defaults")
for agent, s in DB_URLS.items():
    print(f"{agent}: {s['user']}@{s['host']}:{s['port']}/{s['dbname']}")


def test_connections():
    """Checks both databases: can connect, PostGIS present, how many states / cities / demographic rows."""
    for agent, s in DB_URLS.items():
        try:
            with _connect(s) as conn, conn.cursor() as cur:
                cur.execute("SELECT PostGIS_Version(), (SELECT COUNT(*) FROM states), "
                            "(SELECT COUNT(*) FROM cities), (SELECT COUNT(*) FROM state_demographics)")
                v, n_states, n_cities, n_demo = cur.fetchone()
            print(f"OK   {agent}: PostGIS {v.split()[0]} · {n_states} states · {n_cities} cities · {n_demo} demographic rows")
        except Exception as exc:
            print(f"FAIL {agent}: {str(exc).strip()[:300]}")

.env found: D:\GEO-KQML_RESEARCH\kqml-geo-evaluation\.env
Agent-001: postgres@localhost:5433/agent_1_db
Agent-002: postgres@localhost:5433/agent_2_db


## 1. Load the asks

In [41]:
def order_key(qid):
    m = re.match(r"t(\d+)$", qid)
    return (0, int(m.group(1))) if m else (1, qid)


files = sorted(f for f in DATA_DIR.glob("evaluation_per_model_runs-*/evaluation_*_runs-*.csv")
               if "summary" not in f.name)
df = pd.concat([pd.read_csv(f, encoding="utf-8-sig") for f in files], ignore_index=True)
df["model"] = pd.Categorical(df["model"], categories=MODELS, ordered=True)
df["answer"] = df["answer_view"].apply(json.loads)
QIDS = sorted(df["qid"].unique(), key=order_key)

checks = {
    "4 000 asks": len(df) == 4000,
    "100 questions": len(QIDS) == 100,
    "40 asks per question": (df.groupby("qid").size() == 40).all(),
    "same fingerprint = same answer": (df.groupby("fingerprint")["answer_view"].nunique() == 1).all(),
}
for name, ok in checks.items():
    print(("OK   " if ok else "FAIL ") + name)

OK   4 000 asks
OK   100 questions
OK   40 asks per question
OK   same fingerprint = same answer


## 2. Review tools (run once)

In [42]:
# ---------- reading answers
def answer_items(answer):
    """The answer as a list of (name, detail) items, or None if it is not a list."""
    if answer.get("matches") is not None:
        return [(m["name"], m.get("wkt")) if isinstance(m, dict) else (m, None) for m in answer["matches"]]
    if answer.get("geometries") is not None:
        return [(g["entity_name"], g.get("wkt")) for g in answer["geometries"]]
    if answer.get("targets") is not None:
        return [(t["name"], f"meets_zone={t.get('meets_zone')}") for t in answer["targets"]]
    data = answer.get("data")
    recs = []
    if isinstance(data, dict):
        recs = [(g, r) for g in ("complete", "partial", "missing") for r in data.get(g, [])]
    elif isinstance(data, list):
        recs = [("complete", r) for r in data]
    if recs:
        return [(f"{r.get('spatial')} {r.get('year')}",
                 f"{g}: " + ", ".join(f"{k}={v}" for k, v in r.items() if k not in ("spatial", "year")))
                for g, r in recs]
    return None


LIST_KEYS = ("matches", "geometries", "targets", "data")


def answer_short(answer):
    items = answer_items(answer)
    if items is None:
        return json.dumps({k: v for k, v in answer.items() if k != "type"}, ensure_ascii=False)[:200]
    names = ", ".join(str(i[0]) for i in items[:10])
    return f"{len(items)} item(s): {names}" + (" …" if len(items) > 10 else "")


def given_by(group):
    return "; ".join(f"{MODEL_LABEL[m]} / {a.replace('Agent-00', 'A')} ({len(g)})"
                     for (m, a), g in group.groupby(["model", "agent_name"], observed=True))


# ---------- distinct answers per question
variants = []
for qid, asks in df.groupby("qid"):
    for no, (fp, g) in enumerate(sorted(asks.groupby("fingerprint"), key=lambda t: -len(t[1])), start=1):
        variants.append({"qid": qid, "answer_no": no, "fingerprint": fp, "asks": len(g),
                         "given_by": given_by(g), "status": g["answer_status"].iloc[0],
                         "answer": answer_short(g["answer"].iloc[0]),
                         "verdict_in_data": ", ".join(sorted(set(g["answer_verdict"].fillna("unreviewed"))))})
variants = pd.DataFrame(variants)
print(f"{len(QIDS)} questions, {len(variants)} distinct answers, "
      f"{(variants.groupby('qid').size() > 1).sum()} questions where the answers differ")


def _answer_of(qid, no):
    fp = variants[(variants["qid"] == qid) & (variants["answer_no"] == no)]["fingerprint"].iloc[0]
    return fp, df[df["fingerprint"] == fp]["answer"].iloc[0]


# ---------- saved reviews
def load_reviews():
    if REVIEW_FILE.exists():
        r = pd.read_csv(REVIEW_FILE, encoding="utf-8-sig", dtype=str).fillna("")
        return {(row.qid, row.fingerprint): row._asdict() for row in r.itertuples(index=False)}
    return {}


reviews = load_reviews()
print(f"{len(reviews)} answers already reviewed")


# ---------- step 1: answers
def show(qid):
    a = df[df["qid"] == qid].iloc[0]
    n = QIDS.index(qid) + 1
    display(Markdown(f"**Q{n} · {qid} · {a['expected_category']}** — {a['question']}"))
    v = variants[variants["qid"] == qid].copy()
    v["your_verdict"] = [reviews.get((qid, fp), {}).get("verdict", "") for fp in v["fingerprint"]]
    display(v.drop(columns=["qid", "fingerprint"]).set_index("answer_no"))
    if len(v) == 1:
        print("All 40 asks give the same answer.")


def answer(qid, no=1, rows=80):
    """The full answer: items as a table, other fields (totals, verdict, result shape …) below."""
    _, a = _answer_of(qid, no)
    items = answer_items(a)
    print(f"answer {no}: status {a.get('status')}, type {a.get('type')}")
    if items is not None:
        with pd.option_context("display.max_rows", rows):
            display(pd.DataFrame(items, columns=["item", "detail"]))
    rest = {k: v for k, v in a.items() if k not in LIST_KEYS + ("status", "type") and v not in (None, [], {})}
    if rest:
        print(json.dumps(rest, ensure_ascii=False, indent=1)[:3000])


def diff(qid):
    n = variants[variants["qid"] == qid]["answer_no"].max()
    if n == 1:
        print("Only one answer – nothing to compare.")
        return
    _, base = _answer_of(qid, 1)
    bi = answer_items(base)
    for no in range(2, n + 1):
        _, other = _answer_of(qid, no)
        oi = answer_items(other)
        print(f"answer 1 vs answer {no}")
        if bi is None or oi is None:
            print(f"   answer 1:  {answer_short(base)}\n   answer {no}:  {answer_short(other)}")
            continue
        only1, only2 = sorted(set(bi) - set(oi), key=str), sorted(set(oi) - set(bi), key=str)
        print(f"   only in answer 1 ({len(only1)}):", "; ".join(f"{a} [{b}]" for a, b in only1[:40]))
        print(f"   only in answer {no} ({len(only2)}):", "; ".join(f"{a} [{b}]" for a, b in only2[:40]))
        d1, d2 = len(bi) - len(set(bi)), len(oi) - len(set(oi))
        if d1 or d2:
            print(f"   exact duplicates: answer 1 has {d1}, answer {no} has {d2}")


# ---------- step 2: SQL on both databases
REF = re.compile(r"\{(state|city):([^}]+)\}")
_geom_cache = {}
Q, RESULTS = {}, {}


def _connect(settings):
    import psycopg2
    conn = psycopg2.connect(**settings) if isinstance(settings, dict) else psycopg2.connect(settings)
    conn.set_session(readonly=True)
    return conn


def _ref_geometry(kind, name):
    """WKT of a state or city, looked up in whichever database has it (rows with an empty shape are skipped)."""
    key = (kind, name)
    if key not in _geom_cache:
        table, col, geo = ("states", "state_name", "geo_shape") if kind == "state" else ("cities", "city_name", "centroid")
        for agent, url in DB_URLS.items():
            with _connect(url) as conn, conn.cursor() as cur:
                cur.execute(f"SELECT ST_AsText({geo}) FROM {table} WHERE {col} = %s AND {geo} IS NOT NULL", (name,))
                rows = cur.fetchall()
            if rows:
                if len(rows) > 1:
                    print(f"note: {len(rows)} {kind}s named {name} in {agent}, using the first")
                _geom_cache[key] = rows[0][0]
                break
        else:
            raise ValueError(f"{kind} '{name}' not found in any database")
    return _geom_cache[key]


def expand(query):
    """Replaces {state:Name} / {city:Name} with the literal geometry, so a query can use a place
    stored in the other database."""
    return REF.sub(lambda m: f"ST_GeomFromText('{_ref_geometry(m.group(1), m.group(2).strip())}', 4326)", query)


def _run(query):
    """One statement on both databases; rows combined (column `db` = which database returned the row)."""
    expanded = expand(query)
    frames = []
    for agent, url in DB_URLS.items():
        try:
            with _connect(url) as conn, conn.cursor() as cur:
                cur.execute(expanded)
                cols = [c[0] for c in cur.description] if cur.description else []
                rows = cur.fetchmany(SQL_MAX_ROWS) if cols else []
            frames.append(pd.DataFrame(rows, columns=cols).assign(db=agent))
        except Exception as exc:
            print(f"{agent}: ERROR {str(exc).strip()[:300]}")
    if not frames:
        return None
    allrows = pd.concat(frames, ignore_index=True)
    cols = [c for c in allrows.columns if c != "db"]
    if cols and len(allrows):
        return (allrows.astype({c: str for c in cols})
                       .groupby(cols, sort=False, dropna=False)["db"].agg(", ".join).reset_index())
    return allrows


def sql(query, qid=None, show_result=True):
    """Runs each statement (separated by ';') on both databases. Returns the first statement's result."""
    parts = [p for p in re.split(r";\s*\n", query)
             if any(line.strip() and not line.strip().startswith("--") for line in p.splitlines())]
    if not parts:
        print("No SQL for this question.")
        return None
    try:
        import psycopg2  # noqa: F401
    except ImportError:
        print("psycopg2 is not installed: pip install psycopg2-binary")
        return None
    results = []
    for i, part in enumerate(parts, start=1):
        try:
            res = _run(part.strip().rstrip(";"))
        except Exception as exc:
            print(f"ERROR: {exc}")
            res = None
        results.append(res)
        if show_result and res is not None:
            label = f"result {i}: " if len(parts) > 1 else ""
            print(f"{label}{len(res)} row(s)  (db = which database returned the row)")
            with pd.option_context("display.max_rows", 60):
                display(res)
    if qid:
        Q[qid], RESULTS[qid] = query, results
    return results[0]


def compare(qid, result, column):
    """Items in the SQL result vs each distinct answer."""
    if result is None or column not in result:
        print("Nothing to compare (no SQL result).")
        return
    truth = set(result[column])
    for no in variants[variants["qid"] == qid]["answer_no"]:
        _, a = _answer_of(qid, no)
        items = answer_items(a) or []
        got = {i[0] for i in items}
        miss, extra = sorted(truth - got), sorted(got - truth)
        flag = "MATCH" if not miss and not extra else "DIFFERS"
        print(f"answer {no}: {flag} – SQL {len(truth)}, answer {len(got)} distinct names "
              f"({len(items) - len(got)} duplicates)")
        if miss:
            print(f"   in SQL, missing in answer ({len(miss)}):", ", ".join(map(str, miss[:40])))
        if extra:
            print(f"   in answer, not in SQL ({len(extra)}):", ", ".join(map(str, extra[:40])))


# ---------- step 3: your review
def save_reviews():
    out = pd.DataFrame(list(reviews.values()), columns=REVIEW_COLUMNS).sort_values(["qid", "fingerprint"])
    out.to_csv(REVIEW_FILE, index=False, encoding="utf-8-sig")


def review(qid, verdicts, note=""):
    """verdicts = {answer_no: "y"/"n"/"p"}. Answers left at "?" are not saved."""
    v = variants[variants["qid"] == qid].set_index("answer_no")
    sql_text = " || ".join(r.head(30).to_csv(index=False).strip().replace("\n", " / ")
                           for r in RESULTS.get(qid, []) if r is not None)[:3000]
    saved = []
    for no, verdict in verdicts.items():
        verdict = VERDICTS.get(str(verdict).strip().lower())
        if verdict is None:
            print(f"answer {no}: not saved – set it to y / n / p after checking")
            continue
        row = v.loc[no]
        reviews[(qid, row["fingerprint"])] = {
            "qid": qid, "question": df[df["qid"] == qid]["question"].iloc[0],
            "category": df[df["qid"] == qid]["expected_category"].iloc[0],
            "fingerprint": row["fingerprint"], "asks": row["asks"], "given_by": row["given_by"],
            "answer_status": row["status"], "answer_short": row["answer"], "verdict": verdict,
            "sql": Q.get(qid, ""), "sql_result": sql_text, "note": note.strip(),
            "reviewed_at": datetime.now().isoformat(timespec="seconds"),
        }
        saved.append(f"answer {no} = {verdict}")
    if saved:
        save_reviews()
        done = sum(1 for (q, fp) in reviews if q in QIDS)
        print(f"saved {qid}: " + ", ".join(saved) + f"   ({done}/{len(variants)} answers reviewed)")

100 questions, 113 distinct answers, 11 questions where the answers differ
21 answers already reviewed


### 2.0 Test the database connections

Should print two `OK` lines. If not, check `.env` and that both containers are running.

In [43]:
test_connections()

OK   Agent-001: PostGIS 3.6 · 16 states · 1338 cities · 330 demographic rows
OK   Agent-002: PostGIS 3.6 · 16 states · 1782 cities · 160 demographic rows


In [44]:
# ---------- pipeline per model and agent
def pipeline(qid):
    a = df[df["qid"] == qid].copy()
    a["wall_s"] = a["wall_ms"] / 1000

    def uniq(s):
        vals = sorted({str(v) for v in s.dropna() if str(v) not in ("", "nan")})
        return "; ".join(v[:80] for v in vals)

    t = a.groupby(["model", "agent_name"], observed=True).agg(
        category=("category_got", uniq),
        category_ok=("category_ok", "mean"),
        extract_ok=("extract_ok", "mean"),
        extract_diff=("extract_diff", uniq),
        sql_steps=("sql_steps", "median"),
        sql_attempts=("sql_attempts", "mean"),
        sql_errors=("sql_errors", uniq),
        kqml_turns=("kqml_turns", "mean"),
        status=("answer_status", uniq),
        seconds=("wall_s", "mean"),
        tokens=("tokens_total", "mean"),
    ).round(2)
    t = t.rename(index=MODEL_LABEL, level=0)
    print(f"expected category: {a['expected_category'].iloc[0]}   (numbers are means over the 5 reps)")
    display(t)


# ---------- KQML messages, explained
def _parse_messages(text):
    if not isinstance(text, str) or not text.strip():
        return [], False
    try:
        return json.loads(text), False
    except ValueError:                 # the log cell was cut (Excel limit of 32 000 characters)
        head = {}
        for key in ("performative", "sender", "receiver", "reply_with", "in_reply_to", "error"):
            m = re.search(rf'"{key}": "([^"]*)"', text)
            if m:
                head[key] = m.group(1)
        return [head], True


def _years(t):
    t = sorted(t or [])
    if len(t) > 2 and t == list(range(t[0], t[-1] + 1)):
        return f"{t[0]}–{t[-1]}"
    return ", ".join(map(str, t)) or "no year"


def _slot(s):
    attrs = ", ".join(s.get("attributes") or []) or "no attribute"
    place = s.get("spatial")
    place = ", ".join(place) if isinstance(place, list) else place
    return f"{attrs} of {place} ({s.get('entity_type', '')}) for {_years(s.get('temporal'))}"


def _geom_short(g):
    g = str(g or "")
    if g.startswith("POINT"):
        return g[:40]
    kind = re.match(r"(\w+)", g)
    pts = re.search(r"\[(\d+) points\]", g)
    return (kind.group(1) if kind else "geometry") + (f" with {pts.group(1)} points" if pts else "")


def explain_message(msg, truncated=False):
    """One KQML message as plain-language lines."""
    if "error" in msg:
        return [f"ERROR reply: {msg['error']} {msg.get('body', '')} – the peer failed, no data came back."]
    perf = (msg.get("performative") or "?").upper()
    ids = msg.get("reply_with") or msg.get("in_reply_to") or ""
    lines = [f"{perf}  {msg.get('sender')} → {msg.get('receiver')}  (id {ids})"]
    if truncated:
        lines.append("   (message was cut in the log file – only the header could be read)")
        return lines
    c = msg.get("content", {})
    is_ask = perf == "ASK"
    for s in c.get("missing_slots", []):
        lines.append(("   asks for:        " if is_ask else "   still missing:   ") + _slot(s))
    for s in c.get("found_slots", []):
        recs = s.get("data", [])
        vals = "; ".join(", ".join(f"{k}={v}" for k, v in r.items()) for r in recs[:6])
        lines.append(f"   found:           {_slot(s)} → {len(recs)} record(s): {vals}" + (" …" if len(recs) > 6 else ""))
    for g in c.get("missing_geometries", []):
        lines.append(("   asks for shape:  " if is_ask else "   shape not found: ")
                     + f"{g.get('entity_type')} {g.get('spatial_entity')}")
    found_g = c.get("found_geometries", [])
    for g in found_g[:8]:
        lines.append(f"   found shape:     {g.get('entity_type')} {g.get('spatial_entity')}: {_geom_short(g.get('geometry'))}")
    if len(found_g) > 8:
        lines.append(f"   … {len(found_g) - 8} more shapes ({len(found_g)} in total)")
    q = c.get("spatial_query")
    if q:
        ex = q.get("exclude") or []
        plural = {"city": "cities", "state": "states"}.get(q.get("target_entity"), q.get("target_entity"))
        lines.append(f"   spatial query:   all {plural} '{q.get('topic')}' a {_geom_short(q.get('geometry'))}, "
                     f"excluding {len(ex)} name(s) the sender already has" + (f" (e.g. {', '.join(ex[:5])})" if ex else ""))
    if not is_ask:
        found = bool(c.get("found_slots") or c.get("found_geometries"))
        missing = bool(c.get("missing_slots") or c.get("missing_geometries"))
        status = "partial-complete" if found and missing else "complete" if found else "not-found" if missing else "nothing new found (empty reply)"
        lines.append(f"   → status of this reply: {status}")
    return lines


def _norm(text):
    """Message text without request ids and token counts, so equal exchanges group together."""
    msgs, cut = _parse_messages(text)
    if cut:
        return re.sub(r'"(reply_with|in_reply_to)": "[^"]*"', "", str(text))
    clean = [{k: v for k, v in m.items() if k not in ("reply_with", "in_reply_to", "metadata")} for m in msgs]
    return json.dumps(clean, sort_keys=True, ensure_ascii=False)


def _tokens(text):
    msgs, _ = _parse_messages(text)
    return [m.get("metadata", {}).get("token_usage") for m in msgs if (m.get("metadata") or {}).get("token_usage")]


def kqml(qid, raw=False, max_exchanges=6):
    """The different KQML exchanges of a question: who had them, and what the messages say."""
    a = df[df["qid"] == qid]
    no_kqml = a[a["kqml_turns"].fillna(0) == 0]
    if len(no_kqml):
        print(f"No KQML – answered from its own database: {given_by(no_kqml)}\n")
    k = a[a["kqml_turns"].fillna(0) > 0].copy()
    if k.empty:
        print("None of the 40 asks used KQML.")
        return
    k["key"] = [_norm(s) + "||" + _norm(r) for s, r in zip(k["kqml_sent"], k["kqml_received"])]
    groups = sorted(k.groupby("key"), key=lambda t: -len(t[1]))
    print(f"{len(groups)} different KQML exchange(s) (request ids and token counts ignored)\n")
    for i, (_, g) in enumerate(groups[:max_exchanges], start=1):
        sent, recv = g["kqml_sent"].iloc[0], g["kqml_received"].iloc[0]
        print(f"=== exchange {i} · {len(g)} ask(s) · {given_by(g)}")
        sent_msgs, t1 = _parse_messages(sent)
        recv_msgs, t2 = _parse_messages(recv)
        turns = max(len(sent_msgs), len(recv_msgs))
        for turn in range(turns):
            if turns > 1:
                print(f"-- turn {turn + 1}")
            if turn < len(sent_msgs):
                print("\n".join(explain_message(sent_msgs[turn], t1)))
            if turn < len(recv_msgs):
                print("\n".join(explain_message(recv_msgs[turn], t2)))
        toks = [t for r in g["kqml_received"] for t in _tokens(r)]
        if toks:
            print(f"   peer LLM tokens per reply: {min(toks)}–{max(toks)}")
        if raw:
            print("raw ask: ", str(sent)[:1500])
            print("raw tell:", str(recv)[:1500])
        print()
    if len(groups) > max_exchanges:
        print(f"… {len(groups) - max_exchanges} more – kqml('{qid}', max_exchanges=20)")

### 2.1 Overview – where you are

Re-run any time: one row per question with the number of distinct answers and how many you have reviewed.

In [45]:
ov = variants.assign(reviewed=[(q, fp) in reviews for q, fp in zip(variants["qid"], variants["fingerprint"])])
ov = ov.groupby("qid").agg(answers=("answer_no", "size"), reviewed=("reviewed", "sum")).reindex(QIDS)
ov.insert(0, "Q", range(1, len(ov) + 1))
ov["question"] = [df[df["qid"] == q]["question"].iloc[0] for q in ov.index]
print(f"questions fully reviewed: {(ov['reviewed'] == ov['answers']).sum()} / {len(ov)}")
with pd.option_context("display.max_rows", 120):
    display(ov)

questions fully reviewed: 20 / 100


,Q,answers,reviewed,question
qid,,,,
t001,1,1,1,What was the population of Brandenburg in 2018?
t002,2,1,1,How many marriages took place in Sachsen-Anhalt in 2016?
t003,3,1,1,Give me the live births in Rheinland-Pfalz for 2019.
t004,4,1,1,Population of Hamburg and Bremen in 2020
t005,5,1,1,Show the marriages in Bayern from 2016 to 2019.
t006,6,1,1,How many babies were born in Schleswig-Holstein in 2017 and 2018?
t007,7,1,1,"Population, marriages and live births of Saarland in 2015"
t008,8,1,1,What was the population of every German state in 2019?
t009,9,1,1,How many inhabitants did Lower Saxony have in 2021?


## 3. How to read the pipeline, the KQML messages and the SQL

### 3.1 How the system answers a question

1. **Classify** – the LLM decides the query type (DATA_LOOKUP, SPATIAL_ADJACENCY, …) → `category`, `category_ok`.
2. **Extract** – the LLM pulls out places, years and attributes → `extract_ok`, `extract_diff`.
3. **Local SQL** – the agent writes and runs SQL on **its own** database, which holds only part of the data →
   `sql_steps`, `sql_attempts`, `sql_errors`.
4. **Gap detection** – what it could not find becomes a *missing slot*.
5. **KQML ask → tell** – it asks the other agent for the missing slots; the peer looks them up and replies → `kqml_turns`.
6. **Merge** – local and peer results become the final answer → `status`.

When an answer is wrong, the cells show where: wrong category or extraction (steps 1–2, `pipeline`), failed SQL
(step 3, `sql_errors`), a missing or failed KQML reply (step 5, `kqml`), or the merge (step 6, e.g. duplicates).

### 3.2 Reading a KQML message

| field | meaning |
|---|---|
| `performative` | the speech act: **ask** = request for data, **tell** = reply with data |
| `sender` / `receiver` | which agent sends to which |
| `reply_with` / `in_reply_to` | request id; a tell's `in_reply_to` equals the ask's `reply_with`, which pairs them |
| `language`, `encoding`, `ontology` | the agreed vocabulary: GeoKQML in JSON, ontology `geo-missingness-v2` |
| `missing_slots` | in an **ask**: what the sender is missing (place, years, attributes). In a **tell**: what the peer could not find either |
| `found_slots` | in a **tell**: the data the peer found, with its records |
| `missing_geometries` / `found_geometries` | the same for shapes and points (ask for a city, get `POINT(…)` back) |
| `spatial_query` | asks the peer to run a spatial filter itself: `topic` (e.g. Within), `geometry` (the search area), `target_entity` (city / state) and `exclude` (names the sender already has) |
| `metadata.token_usage` | tokens the peer's LLM used to answer |
| `error` / `body` | the peer failed (e.g. HTTP 500) – nothing came back |

A tell's status follows from its slots: only found → **complete**, found and missing → **partial-complete**,
only missing → **not-found**.

### 3.3 Reading the check SQL

The check queries run on **both** databases and the rows are combined (column `db` = where a row came from).
`{state:Hessen}` and `{city:Kassel}` are placeholders: the notebook fetches that shape from whichever database
has it and puts it into the query, so a city in one database can be compared with states in the other.

| function | meaning |
|---|---|
| `ST_DWithin(a::geography, b::geography, 25000)` | true when a and b are less than 25 000 **metres** apart (`::geography` = measured on the earth, in metres) |
| `ST_Distance(a::geography, b::geography) / 1000` | distance in km (0 when a lies inside b) |
| `ST_Within(point, shape)` | the point lies inside the shape |
| `ST_Touches(a, b)` | the shapes share a border but do not overlap = neighbours |
| `ST_Intersects(a, b)` | the shapes touch or overlap |
| `ST_Union`, `ST_Intersection`, `ST_Difference`, `ST_SymDifference` | merge · common part · a minus b · parts in only one of them |
| `ST_Centroid`, `ST_X`, `ST_Y` | centre point of a shape · its longitude (east–west) · its latitude (north–south) |
| `ST_XMin/XMax/YMin/YMax` | edges of the shape's bounding box |
| `ST_GeometryType`, `ST_NPoints`, `ST_Area` | type (POLYGON, POINT …) · number of vertices · area (km² after `/ 1e6`) |
| `state_demographics JOIN states USING (state_id)` | the yearly numbers of a state, found by its name |

## Q1 · t001 · DATA_LOOKUP

**What was the population of Brandenburg in 2018?**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [10]:
show("t001")
answer("t001", 1)

**Q1 · t001 · DATA_LOOKUP** — What was the population of Brandenburg in 2018?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,1 item(s): Brandenburg 2018,correct,correct


All 40 asks give the same answer.
answer 1: status complete, type DATA_LOOKUP


,item,detail
0,Brandenburg 2018,complete: population=2511917


In [11]:
pipeline("t001")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.63  3739.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.35  3739.0
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     6.62  6222.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.96  6222.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.64  3918.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.09  3918.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.11  3913.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.50  3913.0

In [12]:
kqml("t001")          # kqml("t001", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [13]:
Q["t001"] = """
SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Brandenburg')
  AND d.stat_year IN (2018)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t001"], qid="t001")

1 row(s)  (db = which database returned the row)


,state_name,stat_year,population,db
0,Brandenburg,2018,2511917,"Agent-001, Agent-002"


In [37]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t001", {1: "y"}, note="""
correct
""")

saved t001: answer 1 = correct   (1/113 answers reviewed)


## Q2 · t002 · DATA_LOOKUP

**How many marriages took place in Sachsen-Anhalt in 2016?**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [38]:
show("t002")
answer("t002", 1)

**Q2 · t002 · DATA_LOOKUP** — How many marriages took place in Sachsen-Anhalt in 2016?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,1 item(s): Sachsen-Anhalt 2016,correct,


All 40 asks give the same answer.
answer 1: status complete, type DATA_LOOKUP


,item,detail
0,Sachsen-Anhalt 2016,complete: marriages=10465


In [39]:
pipeline("t002")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.87  3753.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.74  6035.4
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.98  6263.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.29  9945.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.82  3933.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    17.84  6262.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.48  3928.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    19.73  6257.0

In [40]:
kqml("t002")          # kqml("t002", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)
ASK  Agent-2 → Agent-1  (id req-201)
   asks for:        marriages of Sachsen-Anhalt (state) for 2016
TELL  Agent-1 → Agent-2  (id req-201)
   found:           marriages of Sachsen-Anhalt (state) for 2016 → 1 record(s): spatial=Sachsen-Anhalt, year=2016, marriages=10465
   → status of this reply: complete
   peer LLM tokens per reply: 1383–2241



In [41]:
Q["t002"] = """
SELECT s.state_name, d.stat_year, d.marriages
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Sachsen-Anhalt')
  AND d.stat_year IN (2016)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t002"], qid="t002")

1 row(s)  (db = which database returned the row)


,state_name,stat_year,marriages,db
0,Sachsen-Anhalt,2016,10465,Agent-001


In [42]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t002", {1: "y"}, note="""

""")

saved t002: answer 1 = correct   (2/113 answers reviewed)


## Q3 · t003 · DATA_LOOKUP

**Give me the live births in Rheinland-Pfalz for 2019.**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [43]:
show("t003")
answer("t003", 1)

**Q3 · t003 · DATA_LOOKUP** — Give me the live births in Rheinland-Pfalz for 2019.

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,1 item(s): Rheinland-Pfalz 2019,correct,


All 40 asks give the same answer.
answer 1: status complete, type DATA_LOOKUP


,item,detail
0,Rheinland-Pfalz 2019,complete: live_births=19020


In [44]:
pipeline("t003")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.92  3762.2
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.17  3762.2
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.83  6261.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.11  6261.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    12.09  3952.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.55  3952.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    12.28  3947.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.72  3947.0

In [45]:
kqml("t003")          # kqml("t003", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [47]:
Q["t003"] = """
SELECT s.state_name, d.stat_year, d.live_births
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Rheinland-Pfalz')
  AND d.stat_year IN (2019)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t003"], qid="t003")

1 row(s)  (db = which database returned the row)


,state_name,stat_year,live_births,db
0,Rheinland-Pfalz,2019,19020,"Agent-001, Agent-002"


In [14]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t003", {1: "y"}, note="""

""")

saved t003: answer 1 = correct   (17/113 answers reviewed)


## Q4 · t004 · DATA_LOOKUP

**Population of Hamburg and Bremen in 2020**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [49]:
show("t004")
answer("t004", 1)

**Q4 · t004 · DATA_LOOKUP** — Population of Hamburg and Bremen in 2020

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,"2 item(s): Hamburg 2020, Bremen 2020",correct,


All 40 asks give the same answer.
answer 1: status complete, type DATA_LOOKUP


,item,detail
0,Hamburg 2020,complete: population=1852478
1,Bremen 2020,complete: population=680130


In [50]:
pipeline("t004")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.19  6030.8
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.77  3743.8
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.28  9911.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.18  6237.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    17.48  6257.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.66  3924.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    18.27  6252.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.37  3919.0

In [51]:
kqml("t004")          # kqml("t004", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-436)
   asks for:        population of Hamburg, Bremen (state) for 2020
TELL  Agent-2 → Agent-1  (id req-436)
   found:           population of Hamburg, Bremen (state) for 2020 → 2 record(s): spatial=Hamburg, year=2020, population=1852478; spatial=Bremen, year=2020, population=680130
   → status of this reply: complete
   peer LLM tokens per reply: 1383–2232



In [52]:
Q["t004"] = """
SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Hamburg', 'Bremen')
  AND d.stat_year IN (2020)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t004"], qid="t004")

2 row(s)  (db = which database returned the row)


,state_name,stat_year,population,db
0,Bremen,2020,680130,Agent-002
1,Hamburg,2020,1852478,Agent-002


In [53]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t004", {1: "y"}, note="""

""")

saved t004: answer 1 = correct   (3/113 answers reviewed)


## Q5 · t005 · DATA_LOOKUP

**Show the marriages in Bayern from 2016 to 2019.**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [54]:
show("t005")
answer("t005", 1)

**Q5 · t005 · DATA_LOOKUP** — Show the marriages in Bayern from 2016 to 2019.

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,partial-complete,"4 item(s): Bayern 2016, Bayern 2017, Bayern 2019, Bayern 2018",correct,


All 40 asks give the same answer.
answer 1: status partial-complete, type DATA_LOOKUP


,item,detail
0,Bayern 2016,complete: marriages=66324
1,Bayern 2017,complete: marriages=66790
2,Bayern 2019,complete: marriages=68501
3,Bayern 2018,missing: marriages=None


In [55]:
pipeline("t005")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns            status  seconds  tokens
model            agent_name                                                                                                                                      
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete     9.15  6078.4
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  partial-complete     9.04  5159.4
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete     9.69  9947.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  partial-complete     8.60  8496.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete    17.23  6338.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  partial-complete    16.09  5400.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete    19.73  6333.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  partial-complete    16.98  5395.0

In [56]:
kqml("t005")          # kqml("t005", raw=True) also prints the raw JSON

2 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-441)
   asks for:        marriages of Bayern (state) for 2016–2019
TELL  Agent-2 → Agent-1  (id req-441)
   still missing:   marriages of Bayern (state) for 2018
   found:           marriages of Bayern (state) for 2016–2019 → 3 record(s): spatial=Bayern, year=2016, marriages=66324; spatial=Bayern, year=2017, marriages=66790; spatial=Bayern, year=2019, marriages=68501
   → status of this reply: partial-complete
   peer LLM tokens per reply: 1404–2251

=== exchange 2 · 20 ask(s) · GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)
ASK  Agent-2 → Agent-1  (id req-206)
   asks for:        marriages of Bayern (state) for 2018
TELL  Agent-1 → Agent-2  (id req-206)
   still missing:   marriages of Bayern (state) for 2018
   → stat

In [57]:
Q["t005"] = """
SELECT s.state_name, d.stat_year, d.marriages
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Bayern')
  AND d.stat_year IN (2016, 2017, 2018, 2019)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t005"], qid="t005")

4 row(s)  (db = which database returned the row)


,state_name,stat_year,marriages,db
0,Bayern,2016,66324.0,Agent-002
1,Bayern,2017,66790.0,Agent-002
2,Bayern,2018,NaN,Agent-002
3,Bayern,2019,68501.0,Agent-002


In [59]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t005", {1: "y"}, note="""

""")

saved t005: answer 1 = correct   (4/113 answers reviewed)


## Q6 · t006 · DATA_LOOKUP

**How many babies were born in Schleswig-Holstein in 2017 and 2018?**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [60]:
show("t006")
answer("t006", 1)

**Q6 · t006 · DATA_LOOKUP** — How many babies were born in Schleswig-Holstein in 2017 and 2018?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,"2 item(s): Schleswig-Holstein 2017, Schleswig-Holstein 2018",correct,


All 40 asks give the same answer.
answer 1: status complete, type DATA_LOOKUP


,item,detail
0,Schleswig-Holstein 2017,complete: live_births=12845
1,Schleswig-Holstein 2018,complete: live_births=12924


In [61]:
pipeline("t006")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.40  3774.6
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.55  6066.6
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.90  6277.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     8.96  9963.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.07  3980.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    17.55  6335.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.91  3975.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    19.60  6330.0

In [62]:
kqml("t006")          # kqml("t006", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)
ASK  Agent-2 → Agent-1  (id req-211)
   asks for:        live_births of Schleswig-Holstein (state) for 2017, 2018
TELL  Agent-1 → Agent-2  (id req-211)
   found:           live_births of Schleswig-Holstein (state) for 2017, 2018 → 2 record(s): spatial=Schleswig-Holstein, year=2017, live_births=12845; spatial=Schleswig-Holstein, year=2018, live_births=12924
   → status of this reply: complete
   peer LLM tokens per reply: 1395–2247



In [63]:
Q["t006"] = """
SELECT s.state_name, d.stat_year, d.live_births
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Schleswig-Holstein')
  AND d.stat_year IN (2017, 2018)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t006"], qid="t006")

2 row(s)  (db = which database returned the row)


,state_name,stat_year,live_births,db
0,Schleswig-Holstein,2017,12845,Agent-001
1,Schleswig-Holstein,2018,12924,Agent-001


In [64]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t006", {1: "y"}, note="""

""")

saved t006: answer 1 = correct   (5/113 answers reviewed)


## Q7 · t007 · DATA_LOOKUP

**Population, marriages and live births of Saarland in 2015**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [65]:
show("t007")
answer("t007", 1)

**Q7 · t007 · DATA_LOOKUP** — Population, marriages and live births of Saarland in 2015

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,1 item(s): Saarland 2015,correct,


All 40 asks give the same answer.
answer 1: status complete, type DATA_LOOKUP


,item,detail
0,Saarland 2015,"complete: live_births=3957, marriages=4861, population=995597"


In [66]:
pipeline("t007")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.71  3782.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.68  6087.6
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.04  6270.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.59  9958.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.88  3964.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    18.97  6317.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.95  3959.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    19.94  6312.0

In [67]:
kqml("t007")          # kqml("t007", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)
ASK  Agent-2 → Agent-1  (id req-216)
   asks for:        population, marriages, live_births of Saarland (state) for 2015
TELL  Agent-1 → Agent-2  (id req-216)
   found:           population, marriages, live_births of Saarland (state) for 2015 → 1 record(s): spatial=Saarland, year=2015, population=995597, marriages=4861, live_births=3957
   → status of this reply: complete
   peer LLM tokens per reply: 1409–2261



In [68]:
Q["t007"] = """
SELECT s.state_name, d.stat_year, d.live_births, d.marriages, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Saarland')
  AND d.stat_year IN (2015)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t007"], qid="t007")

1 row(s)  (db = which database returned the row)


,state_name,stat_year,live_births,marriages,population,db
0,Saarland,2015,3957,4861,995597,Agent-001


In [69]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t007", {1: "y"}, note="""

""")

saved t007: answer 1 = correct   (6/113 answers reviewed)


## Q8 · t008 · DATA_LOOKUP

**What was the population of every German state in 2019?**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [70]:
show("t008")
answer("t008", 1)

**Q8 · t008 · DATA_LOOKUP** — What was the population of every German state in 2019?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,partial-complete,"16 item(s): Baden-Württemberg 2019, Bayern 2019, Nordrhein-Westfalen 2019, Hamburg 2019, Thüringen 2019, Brandenburg...",correct,


All 40 asks give the same answer.
answer 1: status partial-complete, type DATA_LOOKUP


,item,detail
0,Baden-Württemberg 2019,complete: population=11100394
1,Bayern 2019,complete: population=13124737
2,Nordrhein-Westfalen 2019,complete: population=17947221
3,Hamburg 2019,complete: population=1847253
4,Thüringen 2019,complete: population=2133378
5,Brandenburg 2019,complete: population=2521893
6,Schleswig-Holstein 2019,complete: population=2903773
7,Sachsen 2019,complete: population=4071971
8,Rheinland-Pfalz 2019,complete: population=4093903
9,Hessen 2019,complete: population=6288080


In [71]:
pipeline("t008")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns            status  seconds   tokens
model            agent_name                                                                                                                                       
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete     9.51   6217.6
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete     9.96   6245.2
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete     9.14  10305.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete     9.83  10364.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete    17.68   6514.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete    20.37   6544.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete    20.55   6509.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  partial-complete    18.79   6539.0

In [72]:
kqml("t008")          # kqml("t008", raw=True) also prints the raw JSON

2 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-446)
   asks for:        population of Baden-Württemberg, Bayern, Berlin, Bremen, Hamburg, Mecklenburg-Vorpommern, Saarland, Sachsen-Anhalt (state) for 2019
TELL  Agent-2 → Agent-1  (id req-446)
   still missing:   population of Berlin, Mecklenburg-Vorpommern, Saarland, Sachsen-Anhalt (state) for 2019
   found:           population of Baden-Württemberg, Bayern, Berlin, Bremen, Hamburg, Mecklenburg-Vorpommern, Saarland, Sachsen-Anhalt (state) for 2019 → 4 record(s): spatial=Baden-Württemberg, year=2019, population=11100394; spatial=Bayern, year=2019, population=13124737; spatial=Bremen, year=2019, population=681202; spatial=Hamburg, year=2019, population=1847253
   → status of this reply: partial-complete
   peer LLM tokens per reply: 1434–2342

=== exchange 2 · 20 a

In [73]:
Q["t008"] = """
SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen')
  AND d.stat_year IN (2019)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t008"], qid="t008")

17 row(s)  (db = which database returned the row)


,state_name,stat_year,population,db
0,Brandenburg,2019,2521893.0,Agent-001
1,Hessen,2019,6288080.0,"Agent-001, Agent-002"
2,Mecklenburg-Vorpommern,2019,NaN,Agent-001
3,Niedersachsen,2019,7993608.0,"Agent-001, Agent-002"
4,Nordrhein-Westfalen,2019,17947221.0,"Agent-001, Agent-002"
5,Rheinland-Pfalz,2019,4093903.0,"Agent-001, Agent-002"
6,Saarland,2019,NaN,Agent-001
7,Sachsen,2019,4071971.0,Agent-001
8,Sachsen-Anhalt,2019,NaN,Agent-001
9,Schleswig-Holstein,2019,2903773.0,Agent-001


In [74]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t008", {1: "y"}, note="""

""")

saved t008: answer 1 = correct   (7/113 answers reviewed)


## Q9 · t009 · DATA_LOOKUP

**How many inhabitants did Lower Saxony have in 2021?**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [75]:
show("t009")
answer("t009", 1)

**Q9 · t009 · DATA_LOOKUP** — How many inhabitants did Lower Saxony have in 2021?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,1 item(s): Niedersachsen 2021,correct,


All 40 asks give the same answer.
answer 1: status complete, type DATA_LOOKUP


,item,detail
0,Niedersachsen 2021,complete: population=8027031


In [76]:
pipeline("t009")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.58  6019.8
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.06  3745.0
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.31  9893.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.95  6235.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    17.41  6253.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.31  3926.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    18.68  6248.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.81  3921.0

In [77]:
kqml("t009")          # kqml("t009", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-451)
   asks for:        population of Niedersachsen (state) for 2021
TELL  Agent-2 → Agent-1  (id req-451)
   found:           population of Niedersachsen (state) for 2021 → 1 record(s): spatial=Niedersachsen, year=2021, population=8027031
   → status of this reply: complete
   peer LLM tokens per reply: 1378–2225



In [78]:
Q["t009"] = """
SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Niedersachsen')
  AND d.stat_year IN (2021)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t009"], qid="t009")

1 row(s)  (db = which database returned the row)


,state_name,stat_year,population,db
0,Niedersachsen,2021,8027031,Agent-002


In [79]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t009", {1: "y"}, note="""

""")

saved t009: answer 1 = correct   (8/113 answers reviewed)


## Q10 · t010 · DATA_LOOKUP

**Marriages in North Rhine-Westphalia in 2022**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [80]:
show("t010")
answer("t010", 1)

**Q10 · t010 · DATA_LOOKUP** — Marriages in North Rhine-Westphalia in 2022

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,1 item(s): Nordrhein-Westfalen 2022,correct,


All 40 asks give the same answer.
answer 1: status complete, type DATA_LOOKUP


,item,detail
0,Nordrhein-Westfalen 2022,complete: marriages=85008


In [81]:
pipeline("t010")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.40  6025.2
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.05  3747.0
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.04  9944.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.30  6258.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    18.66  6284.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.73  3940.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    19.37  6279.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.57  3935.0

In [82]:
kqml("t010")          # kqml("t010", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-456)
   asks for:        marriages of Nordrhein-Westfalen (state) for 2022
TELL  Agent-2 → Agent-1  (id req-456)
   found:           marriages of Nordrhein-Westfalen (state) for 2022 → 1 record(s): spatial=Nordrhein-Westfalen, year=2022, marriages=85008
   → status of this reply: complete
   peer LLM tokens per reply: 1381–2243



In [83]:
Q["t010"] = """
SELECT s.state_name, d.stat_year, d.marriages
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Nordrhein-Westfalen')
  AND d.stat_year IN (2022)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t010"], qid="t010")

1 row(s)  (db = which database returned the row)


,state_name,stat_year,marriages,db
0,Nordrhein-Westfalen,2022,85008,Agent-002


In [84]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t010", {1: "y"}, note="""

""")

saved t010: answer 1 = correct   (9/113 answers reviewed)


## Q11 · t011 · DATA_LOOKUP

**Population of Thuringia in 1995**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [85]:
show("t011")
answer("t011", 1)

**Q11 · t011 · DATA_LOOKUP** — Population of Thuringia in 1995

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,not-found,1 item(s): Thüringen 1995,correct,


All 40 asks give the same answer.
answer 1: status not-found, type DATA_LOOKUP


,item,detail
0,Thüringen 1995,missing: population=None


In [86]:
pipeline("t011")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns     status  seconds  tokens
model            agent_name                                                                                                                               
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  not-found     9.07  5117.6
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  not-found     9.34  6013.8
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  not-found     8.25  8432.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  not-found     9.09  9861.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  not-found    14.99  5334.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  not-found    16.68  6238.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  not-found    15.81  5329.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  not-found    18.76  6233.0

In [87]:
kqml("t011")          # kqml("t011", raw=True) also prints the raw JSON

2 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)
ASK  Agent-2 → Agent-1  (id req-226)
   asks for:        population of Thüringen (state) for 1995
TELL  Agent-1 → Agent-2  (id req-226)
   still missing:   population of Thüringen (state) for 1995
   → status of this reply: not-found
   peer LLM tokens per reply: 1379–2221

=== exchange 2 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-461)
   asks for:        population of Thüringen (state) for 1995
TELL  Agent-2 → Agent-1  (id req-461)
   still missing:   population of Thüringen (state) for 1995
   → status of this reply: not-found
   peer LLM tokens per reply: 1380–2221



In [88]:
Q["t011"] = """
SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Thüringen')
  AND d.stat_year IN (1995)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t011"], qid="t011")

1 row(s)  (db = which database returned the row)


,state_name,stat_year,population,db
0,Thüringen,1995,NaN,Agent-001


In [90]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t011", {1: "y"}, note="""

""")

saved t011: answer 1 = correct   (10/113 answers reviewed)


## Q12 · t012 · DATA_LOOKUP

**How many people live in Mecklenburg-Vorpommern?**

All 40 asks give the same answer.

**SQL check:** The question names no year: lists every year in the data, so you can judge whether asking for a year (NEEDS_YEAR) is right.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [91]:
show("t012")
answer("t012", 1)

**Q12 · t012 · DATA_LOOKUP** — How many people live in Mecklenburg-Vorpommern?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,rejected,"{""status"": ""rejected""}",correct,


All 40 asks give the same answer.
answer 1: status rejected, type NEEDS_YEAR


In [92]:
pipeline("t012")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     5.51  2367.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     3.97  2367.0
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     4.09  4018.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     3.89  4018.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     6.84  2506.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     6.49  2506.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     7.12  2501.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     7.19  2501.0

In [93]:
kqml("t012")          # kqml("t012", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [96]:
Q["t012"] = """
SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Mecklenburg-Vorpommern')
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t012"], qid="t012")

30 row(s)  (db = which database returned the row)


,state_name,stat_year,population,db
0,Mecklenburg-Vorpommern,1990,NaN,Agent-001
1,Mecklenburg-Vorpommern,1991,NaN,Agent-001
2,Mecklenburg-Vorpommern,1992,NaN,Agent-001
3,Mecklenburg-Vorpommern,1993,NaN,Agent-001
4,Mecklenburg-Vorpommern,1994,NaN,Agent-001
5,Mecklenburg-Vorpommern,1995,NaN,Agent-001
6,Mecklenburg-Vorpommern,1996,NaN,Agent-001
7,Mecklenburg-Vorpommern,1997,NaN,Agent-001
8,Mecklenburg-Vorpommern,1998,NaN,Agent-001
9,Mecklenburg-Vorpommern,1999,NaN,Agent-001


In [98]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t012", {1: "y"}, note="""

""")

saved t012: answer 1 = correct   (11/113 answers reviewed)


## Q13 · t013 · DATA_LOOKUP

**Give me the live births of Berlin**

All 40 asks give the same answer.

**SQL check:** The question names no year: lists every year in the data, so you can judge whether asking for a year (NEEDS_YEAR) is right.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [99]:
show("t013")
answer("t013", 1)

**Q13 · t013 · DATA_LOOKUP** — Give me the live births of Berlin

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,rejected,"{""status"": ""rejected""}",correct,


All 40 asks give the same answer.
answer 1: status rejected, type NEEDS_YEAR


In [100]:
pipeline("t013")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     3.67  2356.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     3.64  2356.0
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     4.18  3992.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     3.83  3992.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     6.49  2487.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     6.57  2487.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     7.64  2482.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     7.17  2482.0

In [101]:
kqml("t013")          # kqml("t013", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [102]:
Q["t013"] = """
SELECT s.state_name, d.stat_year, d.live_births
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Berlin')
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t013"], qid="t013")

16 row(s)  (db = which database returned the row)


,state_name,stat_year,live_births,db
0,Berlin,2010,17090.0,Agent-002
1,Berlin,2011,NaN,Agent-002
2,Berlin,2012,17898.0,Agent-002
3,Berlin,2013,18045.0,Agent-002
4,Berlin,2014,19021.0,Agent-002
5,Berlin,2015,19614.0,Agent-002
6,Berlin,2016,20995.0,Agent-002
7,Berlin,2017,20723.0,Agent-002
8,Berlin,2018,20612.0,Agent-002
9,Berlin,2019,NaN,Agent-002


In [103]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t013", {1: "y"}, note="""

""")

saved t013: answer 1 = correct   (12/113 answers reviewed)


## Q14 · t014 · DATA_LOOKUP

**What was the combined population of Brandenburg and Sachsen in 2018?**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer. The question asks for a total: add the values and check the answer's total.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [104]:
show("t014")
answer("t014", 1)

**Q14 · t014 · DATA_LOOKUP** — What was the combined population of Brandenburg and Sachsen in 2018?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,"2 item(s): Brandenburg 2018, Sachsen 2018",correct,


All 40 asks give the same answer.
answer 1: status complete, type DATA_LOOKUP


,item,detail
0,Brandenburg 2018,complete: population=2511917
1,Sachsen 2018,complete: population=4077937


{
 "totals": [
  {
   "aggregate": "sum",
   "attribute": "population",
   "missing": [],
   "places": [
    "Brandenburg",
    "Sachsen"
   ],
   "status": "complete",
   "total": 6589854,
   "year": 2018
  }
 ]
}


In [105]:
pipeline("t014")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.01  3758.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.58  6031.0
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.86  6266.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.28  9916.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.65  3944.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    17.74  6263.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.56  3938.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    19.23  6257.0

In [106]:
kqml("t014")          # kqml("t014", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)
ASK  Agent-2 → Agent-1  (id req-231)
   asks for:        population of Sachsen (state) for 2018
TELL  Agent-1 → Agent-2  (id req-231)
   found:           population of Sachsen (state) for 2018 → 1 record(s): spatial=Sachsen, year=2018, population=4077937
   → status of this reply: complete
   peer LLM tokens per reply: 1378–2221



In [107]:
Q["t014"] = """
SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Brandenburg', 'Sachsen')
  AND d.stat_year IN (2018)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t014"], qid="t014")

2 row(s)  (db = which database returned the row)


,state_name,stat_year,population,db
0,Brandenburg,2018,2511917,"Agent-001, Agent-002"
1,Sachsen,2018,4077937,Agent-001


In [109]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t014", {1: "y"}, note="""

""")

saved t014: answer 1 = correct   (13/113 answers reviewed)


## Q15 · t015 · DATA_LOOKUP

**Total marriages in Hessen, Bayern and Baden-Württemberg in 2017**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer. The question asks for a total: add the values and check the answer's total.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [110]:
show("t015")
answer("t015", 1)

**Q15 · t015 · DATA_LOOKUP** — Total marriages in Hessen, Bayern and Baden-Württemberg in 2017

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,"3 item(s): Hessen 2017, Baden-Württemberg 2017, Bayern 2017",correct,


All 40 asks give the same answer.
answer 1: status complete, type DATA_LOOKUP


,item,detail
0,Hessen 2017,complete: marriages=29281
1,Baden-Württemberg 2017,complete: marriages=54591
2,Bayern 2017,complete: marriages=66790


{
 "totals": [
  {
   "aggregate": "sum",
   "attribute": "marriages",
   "missing": [],
   "places": [
    "Baden-Württemberg",
    "Bayern",
    "Hessen"
   ],
   "status": "complete",
   "total": 150662,
   "year": 2017
  }
 ]
}


In [111]:
pipeline("t015")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds   tokens
model            agent_name                                                                                                                               
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     9.73   6069.4
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.88   3770.4
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete     8.96  10017.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.09   6308.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    17.70   6340.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.07   3978.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           2.0                    1.0  complete    19.38   6334.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    12.52   3972.0

In [112]:
kqml("t015")          # kqml("t015", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-466)
   asks for:        marriages of Hessen, Bayern, Baden-Württemberg (state) for 2017
TELL  Agent-2 → Agent-1  (id req-466)
   found:           marriages of Hessen, Bayern, Baden-Württemberg (state) for 2017 → 3 record(s): spatial=Hessen, year=2017, marriages=29281; spatial=Bayern, year=2017, marriages=66790; spatial=Baden-Württemberg, year=2017, marriages=54591
   → status of this reply: complete
   peer LLM tokens per reply: 1393–2259



In [113]:
Q["t015"] = """
SELECT s.state_name, d.stat_year, d.marriages
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Hessen', 'Bayern', 'Baden-Württemberg')
  AND d.stat_year IN (2017)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t015"], qid="t015")

4 row(s)  (db = which database returned the row)


,state_name,stat_year,marriages,db
0,Hessen,2017,NaN,Agent-001
1,Baden-Württemberg,2017,54591,Agent-002
2,Bayern,2017,66790,Agent-002
3,Hessen,2017,29281,Agent-002


In [114]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t015", {1: "y"}, note="""

""")

saved t015: answer 1 = correct   (14/113 answers reviewed)


## Q16 · t016 · DATA_LOOKUP

**How many people lived in Niedersachsen and Bremen together from 2016 to 2018?**

All 40 asks give the same answer.

**SQL check:** Compare each value with the answer. The question asks for a total: add the values and check the answer's total.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [115]:
show("t016")
answer("t016", 1)

**Q16 · t016 · DATA_LOOKUP** — How many people lived in Niedersachsen and Bremen together from 2016 to 2018?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,partial-complete,"6 item(s): Bremen 2016, Bremen 2018, Niedersachsen 2016, Niedersachsen 2017, Niedersachsen 2018, Bremen 2017",correct,


All 40 asks give the same answer.
answer 1: status partial-complete, type DATA_LOOKUP


,item,detail
0,Bremen 2016,complete: population=678753
1,Bremen 2018,complete: population=682986
2,Niedersachsen 2016,complete: population=7945685
3,Niedersachsen 2017,complete: population=7962775
4,Niedersachsen 2018,complete: population=7982448
5,Bremen 2017,missing: population=None


{
 "full_total_years": [
  2015,
  2016,
  2018,
  2019,
  2020,
  2021,
  2022
 ],
 "totals": [
  {
   "aggregate": "sum",
   "attribute": "population",
   "missing": [
    "Bremen"
   ],
   "places": [
    "Bremen",
    "Niedersachsen"
   ],
   "status": "partial-complete",
   "total": null,
   "year": 2017
  },
  {
   "aggregate": "sum",
   "attribute": "population",
   "missing": [],
   "places": [
    "Bremen",
    "Niedersachsen"
   ],
   "status": "complete",
   "total": 8624438,
   "year": 2016
  },
  {
   "aggregate": "sum",
   "attribute": "population",
   "missing": [],
   "places": [
    "Bremen",
    "Niedersachsen"
   ],
   "status": "complete",
   "total": 8665434,
   "year": 2018
  }
 ]
}


In [116]:
pipeline("t016")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns            status  seconds   tokens
model            agent_name                                                                                                                                       
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           4.0                    2.0  partial-complete    17.96  11355.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           2.0                    2.0  partial-complete    14.12   9478.2
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           4.0                    2.0  partial-complete    15.91  18310.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           2.0                    2.0  partial-complete    13.61  15377.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           4.0                    2.0  partial-complete    30.41  11881.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           2.0                    2.0  partial-complete    25.46   9944.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     2.0           4.0                    2.0  partial-complete    34.05  11875.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     1.0           2.0                    2.0  partial-complete    28.37   9938.0

In [117]:
kqml("t016")          # kqml("t016", raw=True) also prints the raw JSON

2 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
-- turn 1
ASK  Agent-1 → Agent-2  (id req-471)
   asks for:        population of Bremen (state) for 2016–2018
TELL  Agent-2 → Agent-1  (id req-471)
   still missing:   population of Bremen (state) for 2017
   found:           population of Bremen (state) for 2016–2018 → 2 record(s): spatial=Bremen, year=2016, population=678753; spatial=Bremen, year=2018, population=682986
   → status of this reply: partial-complete
-- turn 2
ASK  Agent-1 → Agent-2  (id req-472)
   asks for:        population of Niedersachsen (state) for 2011, 2012, 2013, 2014, 2020, 2021, 2022, 2023
   asks for:        population of Bremen (state) for 2011–2023
TELL  Agent-2 → Agent-1  (id req-472)
   still missing:   population of Niedersachsen (state) for 2011, 2012, 2013, 2014, 2023
   still missing:   population of Bremen (sta

In [118]:
Q["t016"] = """
SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Niedersachsen', 'Bremen')
  AND d.stat_year IN (2016, 2017, 2018)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t016"], qid="t016")

7 row(s)  (db = which database returned the row)


,state_name,stat_year,population,db
0,Niedersachsen,2016,7945685.0,"Agent-001, Agent-002"
1,Niedersachsen,2017,7962775.0,Agent-001
2,Niedersachsen,2018,7982448.0,"Agent-001, Agent-002"
3,Bremen,2016,678753.0,Agent-002
4,Bremen,2017,NaN,Agent-002
5,Bremen,2018,682986.0,Agent-002
6,Niedersachsen,2017,NaN,Agent-002


In [119]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t016", {1: "y"}, note="""

""")

saved t016: answer 1 = correct   (15/113 answers reviewed)


## Q17 · t017 · DATA_LOOKUP

**Population of Bavaria in 2030**

⚠ **2 different answers** – see `diff` below.

> GPT-4o mini rejected the question (NEEDS_YEAR) in 4 of 10 asks; all other asks say not-found for Bayern 2030.

**SQL check:** Compare each value with the answer.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [15]:
show("t017")
answer("t017", 1)
diff("t017")

**Q17 · t017 · DATA_LOOKUP** — Population of Bavaria in 2030

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,36,GPT-4o mini / A1 (3); GPT-4o mini / A2 (3); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,not-found,1 item(s): Bayern 2030,correct,
2,4,GPT-4o mini / A1 (2); GPT-4o mini / A2 (2),rejected,"{""status"": ""rejected""}",correct,


answer 1: status not-found, type DATA_LOOKUP


,item,detail
0,Bayern 2030,missing: population=None


answer 1 vs answer 2
   answer 1:  1 item(s): Bayern 2030
   answer 2:  {"status": "rejected"}


In [16]:
pipeline("t017")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok                           extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns               status  seconds  tokens
model            agent_name                                                                                                                                                                   
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         0.6  temporal: expected [2030], got (none)        2.0           2.0                    0.6  not-found; rejected     6.95  4543.0
                 Agent-002   DATA_LOOKUP          1.0         0.6  temporal: expected [2030], got (none)        2.0           2.0                    0.6  not-found; rejected     6.99  4543.0
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                                               2.0           2.0                    1.0            not-found     9.39  9854.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                                               2.0           2.0                    1.0            not-found     8.85  9854.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                                               2.0           2.0                    1.0            not-found    17.21  6227.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                                               2.0           2.0                    1.0            not-found    17.22  6227.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                                               2.0           2.0                    1.0            not-found    18.44  6222.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                                               2.0           2.0                    1.0            not-found    18.83  6222.0

In [17]:
kqml("t017")          # kqml("t017", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (2); GPT-4o mini / A2 (2)

2 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 18 ask(s) · GPT-4o mini / A2 (3); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)
ASK  Agent-2 → Agent-1  (id req-246)
   asks for:        population of Bayern (state) for 2030
TELL  Agent-1 → Agent-2  (id req-246)
   still missing:   population of Bayern (state) for 2030
   → status of this reply: not-found
   peer LLM tokens per reply: 1377–2219

=== exchange 2 · 18 ask(s) · GPT-4o mini / A1 (3); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-481)
   asks for:        population of Bayern (state) for 2030
TELL  Agent-2 → Agent-1  (id req-481)
   still missing:   population of Bayern (state) for 2030
   → status of this reply: not-found
   peer LLM tokens per reply: 1377–2219



In [18]:
Q["t017"] = """
SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Bayern')
  AND d.stat_year IN (2030)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t017"], qid="t017")

0 row(s)  (db = which database returned the row)


,state_name,stat_year,population,db


In [20]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t017", {1: "y", 2: "n"}, note="""
No database has Bayern 2030 (agent_2_db covers 2010–2025, agent_1_db has no Bayern rows),
so not-found is correct. Answer 2: GPT-4o mini dropped the year 2030 during extraction
(4/10 asks) and the agent rejected the question instead of searching.
""")

saved t017: answer 1 = correct, answer 2 = wrong   (19/113 answers reviewed)


## Q18 · t018 · DATA_LOOKUP

**What is the combined number of marriages of Berlin and Brandenburg?**

All 40 asks give the same answer.

**SQL check:** The question names no year: lists every year in the data, so you can judge whether asking for a year (NEEDS_YEAR) is right. The question asks for a total: add the values and check the answer's total.

How the query works:
- `state_demographics JOIN states` = the yearly numbers of the named states

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [21]:
show("t018")
answer("t018", 1)

**Q18 · t018 · DATA_LOOKUP** — What is the combined number of marriages of Berlin and Brandenburg?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,rejected,"{""status"": ""rejected""}",correct,


All 40 asks give the same answer.
answer 1: status rejected, type NEEDS_YEAR


In [25]:
pipeline("t018")

expected category: DATA_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                              
GPT-4o mini      Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     3.69  2372.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     3.98  2372.0
Claude Haiku 5.5 Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     4.00  4032.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     3.91  4032.0
Qwen3-8B         Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     6.81  2509.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     6.78  2509.0
Qwen3-8B + QLoRA Agent-001   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     7.39  2503.0
                 Agent-002   DATA_LOOKUP          1.0         1.0                     0.0           NaN                    0.0  rejected     7.21  2503.0

In [26]:
kqml("t018")          # kqml("t018", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [27]:
Q["t018"] = """
SELECT s.state_name, d.stat_year, d.marriages
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Berlin', 'Brandenburg')
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t018"], qid="t018")

57 row(s)  (db = which database returned the row)


,state_name,stat_year,marriages,db
0,Brandenburg,1990,16814.0,Agent-001
1,Brandenburg,1991,8328.0,Agent-001
2,Brandenburg,1992,NaN,Agent-001
3,Brandenburg,1993,7901.0,Agent-001
4,Brandenburg,1994,8502.0,Agent-001
5,Brandenburg,1995,8775.0,Agent-001
6,Brandenburg,1996,8756.0,Agent-001
7,Brandenburg,1997,8709.0,Agent-001
8,Brandenburg,1998,9266.0,Agent-001
9,Brandenburg,1999,10219.0,Agent-001


In [28]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t018", {1: "y"}, note="""
year is not there
""")

saved t018: answer 1 = correct   (20/113 answers reviewed)


## Q19 · t019 · GEOMETRY_LOOKUP

**Show me the boundary of Niedersachsen**

All 40 asks give the same answer.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

How the query works:
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [29]:
show("t019")
answer("t019", 1)

**Q19 · t019 · GEOMETRY_LOOKUP** — Show me the boundary of Niedersachsen

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,1 item(s): Niedersachsen,correct,


All 40 asks give the same answer.
answer 1: status complete, type GEOMETRY_LOOKUP


,item,detail
0,Niedersachsen,MULTIPOLYGON [14321 vertices #46ceea567e]


In [30]:
pipeline("t019")

expected category: GEOMETRY_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                  
GPT-4o mini      Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.07  3168.8
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.66  3168.8
Claude Haiku 5.5 Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.00  5573.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.08  5573.0
Qwen3-8B         Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     8.62  3273.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     7.88  3273.0
Qwen3-8B + QLoRA Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.58  3278.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.30  3278.0

In [31]:
kqml("t019")          # kqml("t019", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [33]:
Q["t019"] = """
SELECT 'state' AS kind, state_name AS name, ST_GeometryType(geo_shape) AS type,
       NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(geo_shape) p), 0) AS vertices
FROM states WHERE state_name IN ('Niedersachsen')
ORDER BY 1, 2
"""
r = sql(Q["t019"], qid="t019")
compare("t019", r, "name")


1 row(s)  (db = which database returned the row)


,kind,name,type,vertices,db
0,state,Niedersachsen,ST_MultiPolygon,14321,"Agent-001, Agent-002"


answer 1: MATCH – SQL 1, answer 1 distinct names (0 duplicates)


In [34]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t019", {1: "y"}, note="""

""")

saved t019: answer 1 = correct   (21/113 answers reviewed)


## Q20 · t020 · GEOMETRY_LOOKUP

**Give me the shape of Baden-Württemberg as WKT**

All 40 asks give the same answer.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

How the query works:
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [46]:
show("t020")
answer("t020", 1)

**Q20 · t020 · GEOMETRY_LOOKUP** — Give me the shape of Baden-Württemberg as WKT

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,1 item(s): Baden-Württemberg,correct,


All 40 asks give the same answer.
answer 1: status complete, type GEOMETRY_LOOKUP


,item,detail
0,Baden-Württemberg,MULTIPOLYGON [5091 vertices #f829eec3bc]


In [47]:
pipeline("t020")

expected category: GEOMETRY_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                  
GPT-4o mini      Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.47  3177.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.83  3176.6
Claude Haiku 5.5 Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.99  5589.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.95  5589.0
Qwen3-8B         Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     8.40  3294.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     8.43  3294.0
Qwen3-8B + QLoRA Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.05  3299.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.83  3299.0

In [48]:
kqml("t020")          # kqml("t020", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [50]:
Q["t020"] = """
SELECT 'state' AS kind, state_name AS name, ST_GeometryType(geo_shape) AS type,
       NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(geo_shape) p), 0) AS vertices
FROM states WHERE state_name IN ('Baden-Württemberg')
ORDER BY 1, 2
"""
r = sql(Q["t020"], qid="t020")
compare("t020", r, "name")

1 row(s)  (db = which database returned the row)


,kind,name,type,vertices,db
0,state,Baden-Württemberg,ST_MultiPolygon,5091,"Agent-001, Agent-002"


answer 1: MATCH – SQL 1, answer 1 distinct names (0 duplicates)


In [51]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t020", {1: "y"}, note="""

""")

saved t020: answer 1 = correct   (22/113 answers reviewed)


## Q21 · t021 · GEOMETRY_LOOKUP

**What are the coordinates of the city of Leipzig?**

All 40 asks give the same answer.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [52]:
show("t021")
answer("t021", 1)

**Q21 · t021 · GEOMETRY_LOOKUP** — What are the coordinates of the city of Leipzig?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,1 item(s): Leipzig,correct,correct


All 40 asks give the same answer.
answer 1: status complete, type GEOMETRY_LOOKUP


,item,detail
0,Leipzig,POINT 12.3711 51.3394


In [53]:
pipeline("t021")

expected category: GEOMETRY_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                  
GPT-4o mini      Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.73  3177.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.57  3177.0
Claude Haiku 5.5 Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.56  5568.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.89  5568.0
Qwen3-8B         Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     8.54  3269.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     8.12  3269.0
Qwen3-8B + QLoRA Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.98  3274.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.12  3274.0

In [126]:
kqml("t021")          # kqml("t021", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [127]:
Q["t021"] = """
SELECT 'city' AS kind, city_name AS name, ST_AsText(centroid) AS type, 1 AS vertices
FROM cities WHERE city_name IN ('Leipzig')
ORDER BY 1, 2
"""
r = sql(Q["t021"], qid="t021")
compare("t021", r, "name")

1 row(s)  (db = which database returned the row)


,kind,name,type,vertices,db
0,city,Leipzig,POINT(12.3711 51.3394),1,"Agent-001, Agent-002"


answer 1: MATCH – SQL 1, answer 1 distinct names (0 duplicates)


In [128]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t021", {1: "y"}, note="""

""")

saved t021: answer 1 = correct   (16/113 answers reviewed)


## Q22 · t022 · GEOMETRY_LOOKUP

**Return the geometries of Saarland and Rheinland-Pfalz**

All 40 asks give the same answer.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

How the query works:
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [54]:
show("t022")
answer("t022", 1)

**Q22 · t022 · GEOMETRY_LOOKUP** — Return the geometries of Saarland and Rheinland-Pfalz

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,"2 item(s): Rheinland-Pfalz, Saarland",correct,


All 40 asks give the same answer.
answer 1: status complete, type GEOMETRY_LOOKUP


,item,detail
0,Rheinland-Pfalz,MULTIPOLYGON [1449 vertices #c7bf05c106]
1,Saarland,MULTIPOLYGON [426 vertices #cc6c383edb]


In [55]:
pipeline("t022")

expected category: GEOMETRY_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                  
GPT-4o mini      Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.48  3206.4
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.56  3206.6
Claude Haiku 5.5 Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.32  5636.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.13  5636.0
Qwen3-8B         Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.58  3331.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.38  3331.0
Qwen3-8B + QLoRA Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.59  3325.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.39  3325.0

In [56]:
kqml("t022")          # kqml("t022", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [57]:
Q["t022"] = """
SELECT 'state' AS kind, state_name AS name, ST_GeometryType(geo_shape) AS type,
       NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(geo_shape) p), 0) AS vertices
FROM states WHERE state_name IN ('Rheinland-Pfalz', 'Saarland')
ORDER BY 1, 2
"""
r = sql(Q["t022"], qid="t022")
compare("t022", r, "name")

2 row(s)  (db = which database returned the row)


,kind,name,type,vertices,db
0,state,Rheinland-Pfalz,ST_MultiPolygon,1449,"Agent-001, Agent-002"
1,state,Saarland,ST_MultiPolygon,426,"Agent-001, Agent-002"


answer 1: MATCH – SQL 2, answer 2 distinct names (0 duplicates)


In [58]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t022", {1: "y"}, note="""

""")

saved t022: answer 1 = correct   (23/113 answers reviewed)


## Q23 · t023 · GEOMETRY_LOOKUP

**Where is Freiburg im Breisgau located?**

⚠ **2 different answers** – see `diff` below.

> GPT-4o mini rejected the question as UNRELATED in all 10 asks; the other models return Freiburg's point.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [59]:
show("t023")
answer("t023", 1)
diff("t023")

**Q23 · t023 · GEOMETRY_LOOKUP** — Where is Freiburg im Breisgau located?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,30,Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5...,complete,1 item(s): Freiburg im Breisgau,correct,
2,10,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5),rejected,"{""status"": ""rejected""}",correct,


answer 1: status complete, type GEOMETRY_LOOKUP


,item,detail
0,Freiburg im Breisgau,POINT 7.85 47.995


answer 1 vs answer 2
   answer 1:  1 item(s): Freiburg im Breisgau
   answer 2:  {"status": "rejected"}


In [60]:
pipeline("t023")

expected category: GEOMETRY_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                  
GPT-4o mini      Agent-001         UNRELATED          0.0         NaN                     0.0           NaN                    0.0  rejected     2.82  1456.0
                 Agent-002         UNRELATED          0.0         NaN                     0.0           NaN                    0.0  rejected     3.06  1456.0
Claude Haiku 5.5 Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete     9.75  7304.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.40  5725.0
Qwen3-8B         Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete    13.20  4295.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.60  3291.0
Qwen3-8B + QLoRA Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete    14.02  4300.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.85  3296.0

In [61]:
kqml("t023")          # kqml("t023", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 15 ask(s) · Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-486)
   asks for shape:  city Freiburg im Breisgau
TELL  Agent-2 → Agent-1  (id req-486)
   found shape:     city Freiburg im Breisgau: POINT(7.85 47.995)
   → status of this reply: complete
   peer LLM tokens per reply: 1004–1597



In [62]:
Q["t023"] = """
SELECT 'city' AS kind, city_name AS name, ST_AsText(centroid) AS type, 1 AS vertices
FROM cities WHERE city_name IN ('Freiburg im Breisgau')
ORDER BY 1, 2
"""
r = sql(Q["t023"], qid="t023")
compare("t023", r, "name")

1 row(s)  (db = which database returned the row)


,kind,name,type,vertices,db
0,city,Freiburg im Breisgau,POINT(7.85 47.995),1,Agent-002


answer 1: MATCH – SQL 1, answer 1 distinct names (0 duplicates)
answer 2: DIFFERS – SQL 1, answer 0 distinct names (0 duplicates)
   in SQL, missing in answer (1): Freiburg im Breisgau


In [63]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t023", {1: "y", 2: "n"}, note="""
Freiburg im Breisgau exists only in agent_2_db: POINT(7.85 47.995), inside Baden-Württemberg.
Answer 1 matches it; Agent-001 got the point from Agent-002 via KQML (1 turn).
Answer 2: GPT-4o mini classified the question as UNRELATED (10/10 asks) and rejected it -
classification error, no SQL or KQML ran.
""")


saved t023: answer 1 = correct, answer 2 = wrong   (25/113 answers reviewed)


## Q24 · t024 · GEOMETRY_LOOKUP

**Give me the polygon of Bavaria**

All 40 asks give the same answer.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

How the query works:
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [64]:
show("t024")
answer("t024", 1)

**Q24 · t024 · GEOMETRY_LOOKUP** — Give me the polygon of Bavaria

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,1 item(s): Bayern,correct,


All 40 asks give the same answer.
answer 1: status complete, type GEOMETRY_LOOKUP


,item,detail
0,Bayern,MULTIPOLYGON [3868 vertices #64040c6770]


In [65]:
pipeline("t024")

expected category: GEOMETRY_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                  
GPT-4o mini      Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete     7.98  4151.8
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.87  3165.6
Claude Haiku 5.5 Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete     8.81  7146.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.81  5560.0
Qwen3-8B         Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete    13.91  4255.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    10.33  3263.0
Qwen3-8B + QLoRA Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete    13.67  4260.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.14  3268.0

In [66]:
kqml("t024")          # kqml("t024", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-491)
   asks for shape:  state Bayern
TELL  Agent-2 → Agent-1  (id req-491)
   found shape:     state Bayern: MULTIPOLYGON with 3869 points
   → status of this reply: complete
   peer LLM tokens per reply: 985–1586



In [67]:
Q["t024"] = """
SELECT 'state' AS kind, state_name AS name, ST_GeometryType(geo_shape) AS type,
       NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(geo_shape) p), 0) AS vertices
FROM states WHERE state_name IN ('Bayern')
ORDER BY 1, 2
"""
r = sql(Q["t024"], qid="t024")
compare("t024", r, "name")

2 row(s)  (db = which database returned the row)


,kind,name,type,vertices,db
0,state,Bayern,NaN,NaN,Agent-001
1,state,Bayern,ST_MultiPolygon,3868,Agent-002


answer 1: MATCH – SQL 1, answer 1 distinct names (0 duplicates)


In [68]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t024", {1: "y"}, note="""

""")

saved t024: answer 1 = correct   (26/113 answers reviewed)


## Q25 · t025 · GEOMETRY_LOOKUP

**Shapes of Kiel, Lübeck and Rostock**

⚠ **2 different answers** – see `diff` below.

> GPT-4o mini returned one merged MULTIPOINT (as SPATIAL_OPERATION) instead of three separate points – decide whether this format counts.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [69]:
show("t025")
answer("t025", 1)
diff("t025")

**Q25 · t025 · GEOMETRY_LOOKUP** — Shapes of Kiel, Lübeck and Rostock

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,30,Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5...,complete,"3 item(s): Kiel, Lübeck, Rostock",correct,
2,10,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5),complete,"{""result"": {""geometry_type"": ""MULTIPOINT"", ""srid"": 4326, ""wkt"": ""MULTIPOINT [3 vertices #165b8bb953]""}, ""status"": ""c...",correct,


answer 1: status complete, type GEOMETRY_LOOKUP


,item,detail
0,Kiel,POINT 10.1394 54.3233
1,Lübeck,POINT 10.6864 53.8697
2,Rostock,POINT 12.1333 54.0833


answer 1 vs answer 2
   answer 1:  3 item(s): Kiel, Lübeck, Rostock
   answer 2:  {"result": {"geometry_type": "MULTIPOINT", "srid": 4326, "wkt": "MULTIPOINT [3 vertices #165b8bb953]"}, "status": "complete"}


In [105]:
pipeline("t025")

expected category: GEOMETRY_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                    
GPT-4o mini      Agent-001   SPATIAL_OPERATION          0.0         NaN                     2.0           3.0                    1.0  complete    10.43  6792.2
                 Agent-002   SPATIAL_OPERATION          0.0         NaN                     2.0           3.0                    0.0  complete     7.12  5795.0
Claude Haiku 5.5 Agent-001     GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete     8.07  7242.0
                 Agent-002     GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.04  5644.0
Qwen3-8B         Agent-001     GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete    18.46  4341.0
                 Agent-002     GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    13.24  3337.0
Qwen3-8B + QLoRA Agent-001     GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete    17.95  4334.0
                 Agent-002     GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete    11.37  3330.0

In [106]:
kqml("t025")          # kqml("t025", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-496)
   asks for shape:  city Lübeck
   asks for shape:  city Rostock
TELL  Agent-2 → Agent-1  (id req-496)
   found shape:     city Lübeck: POINT(10.6864 53.8697)
   found shape:     city Rostock: POINT(12.1333 54.0833)
   → status of this reply: complete
   peer LLM tokens per reply: 997–1598



In [107]:
Q["t025"] = """
SELECT 'city' AS kind, city_name AS name, ST_AsText(centroid) AS type, 1 AS vertices
FROM cities WHERE city_name IN ('Kiel', 'Lübeck', 'Rostock')
ORDER BY 1, 2
"""
r = sql(Q["t025"], qid="t025")
compare("t025", r, "name")

4 row(s)  (db = which database returned the row)


,kind,name,type,vertices,db
0,city,Kiel,POINT(10.1394 54.3233),1,"Agent-001, Agent-002"
1,city,Rostock,NaN,1,Agent-001
2,city,Lübeck,POINT(10.6864 53.8697),1,Agent-002
3,city,Rostock,POINT(12.1333 54.0833),1,Agent-002


answer 1: MATCH – SQL 3, answer 3 distinct names (0 duplicates)
answer 2: DIFFERS – SQL 3, answer 0 distinct names (0 duplicates)
   in SQL, missing in answer (3): Kiel, Lübeck, Rostock


In [73]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t025", {1: "y", 2: "p"}, note="""
DB: Kiel (10.1394 54.3233), Lübeck (10.6864 53.8697), Rostock (12.1333 54.0833).
Answer 1 matches all three. Agent-001 lacks Lübeck and Rostock's point and fetched both
from Agent-002 with one KQML ask (missing_geometries).
Answer 2 (GPT-4o mini): misclassified as SPATIAL_OPERATION and returned one MULTIPOINT.
Its fingerprint #165b8bb953 equals the three correct points, but the city names are lost -> partly.
""")


saved t025: answer 1 = correct, answer 2 = partly   (28/113 answers reviewed)


## Q26 · t026 · GEOMETRY_LOOKUP

**Show the outline of Schleswig-Holstein and the location of Flensburg**

All 40 asks give the same answer.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

How the query works:
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [92]:
show("t026")
answer("t026", 1)

**Q26 · t026 · GEOMETRY_LOOKUP** — Show the outline of Schleswig-Holstein and the location of Flensburg

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,"2 item(s): Flensburg, Schleswig-Holstein",correct,


All 40 asks give the same answer.
answer 1: status complete, type GEOMETRY_LOOKUP


,item,detail
0,Flensburg,POINT 9.4367 54.7819
1,Schleswig-Holstein,MULTIPOLYGON [27974 vertices #39c75bba4d]


In [93]:
pipeline("t026")

expected category: GEOMETRY_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                  
GPT-4o mini      Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           2.0                    1.0  complete     8.76  5164.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           2.0                    0.0  complete     5.76  4178.0
Claude Haiku 5.5 Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           2.0                    1.0  complete    10.38  8795.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           2.0                    0.0  complete     6.35  7210.0
Qwen3-8B         Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           2.0                    1.0  complete    16.20  5303.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           2.0                    0.0  complete    11.54  4311.0
Qwen3-8B + QLoRA Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           2.0                    1.0  complete    15.78  5297.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           2.0                    0.0  complete    11.97  4305.0

In [94]:
kqml("t026")          # kqml("t026", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-501)
   asks for shape:  city Flensburg
TELL  Agent-2 → Agent-1  (id req-501)
   found shape:     city Flensburg: POINT(9.4367 54.7819)
   → status of this reply: complete
   peer LLM tokens per reply: 986–1585



In [95]:
Q["t026"] = """
SELECT 'state' AS kind, state_name AS name, ST_GeometryType(geo_shape) AS type,
       NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(geo_shape) p), 0) AS vertices
FROM states WHERE state_name IN ('Schleswig-Holstein')
UNION ALL
SELECT 'city' AS kind, city_name AS name, ST_AsText(centroid) AS type, 1 AS vertices
FROM cities WHERE city_name IN ('Flensburg')
ORDER BY 1, 2
"""
r = sql(Q["t026"], qid="t026")
compare("t026", r, "name")

3 row(s)  (db = which database returned the row)


,kind,name,type,vertices,db
0,city,Flensburg,NaN,1,Agent-001
1,state,Schleswig-Holstein,ST_MultiPolygon,27974,"Agent-001, Agent-002"
2,city,Flensburg,POINT(9.4367 54.7819),1,Agent-002


answer 1: MATCH – SQL 2, answer 2 distinct names (0 duplicates)


In [108]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t026", {1: "y"}, note="""
Agent-001 has Flensburg without a point; it got POINT(9.4367 54.7819) from Agent-002 via KQML.
""")


saved t026: answer 1 = correct   (30/113 answers reviewed)


## Q27 · t027 · GEOMETRY_LOOKUP

**Get the geometry of the state of Bremen**

All 40 asks give the same answer.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

How the query works:
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [109]:
show("t027")
answer("t027", 1)

**Q27 · t027 · GEOMETRY_LOOKUP** — Get the geometry of the state of Bremen

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,1 item(s): Bremen,correct,correct


All 40 asks give the same answer.
answer 1: status complete, type GEOMETRY_LOOKUP


,item,detail
0,Bremen,MULTIPOLYGON [156 vertices #e07b4369a5]


In [110]:
pipeline("t027")

expected category: GEOMETRY_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                  
GPT-4o mini      Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.56  3170.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.48  3170.0
Claude Haiku 5.5 Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     4.87  5562.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.03  5562.0
Qwen3-8B         Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.00  3267.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     7.92  3267.0
Qwen3-8B + QLoRA Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.18  3272.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.76  3272.0

In [111]:
kqml("t027")          # kqml("t027", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [112]:
Q["t027"] = """
SELECT 'state' AS kind, state_name AS name, ST_GeometryType(geo_shape) AS type,
       NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(geo_shape) p), 0) AS vertices
FROM states WHERE state_name IN ('Bremen')
ORDER BY 1, 2
"""
r = sql(Q["t027"], qid="t027")
compare("t027", r, "name")

1 row(s)  (db = which database returned the row)


,kind,name,type,vertices,db
0,state,Bremen,ST_MultiPolygon,156,"Agent-001, Agent-002"


answer 1: MATCH – SQL 1, answer 1 distinct names (0 duplicates)


In [113]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t027", {1: "y"}, note="""

""")


saved t027: answer 1 = correct   (30/113 answers reviewed)


## Q28 · t029 · GEOMETRY_LOOKUP

**Boundaries of all German federal states**

⚠ **2 different answers** – see `diff` below.

> Qwen3-8B + QLoRA extracted 'Mecklenburg-Vorpowell' (typo), so it returned 15 of 16 states.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

How the query works:
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [114]:
show("t029")
answer("t029", 1)
diff("t029")

**Q28 · t029 · GEOMETRY_LOOKUP** — Boundaries of all German federal states

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,30,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,"16 item(s): Baden-Württemberg, Bayern, Berlin, Brandenburg, Bremen, Hamburg, Hessen, Mecklenburg-Vorpommern, Nieders...",correct,
2,10,Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5),partial-complete,"16 item(s): Baden-Württemberg, Bayern, Berlin, Brandenburg, Bremen, Hamburg, Hessen, Mecklenburg-Vorpowell, Niedersa...",wrong,


answer 1: status complete, type GEOMETRY_LOOKUP


,item,detail
0,Baden-Württemberg,MULTIPOLYGON [5091 vertices #f829eec3bc]
1,Bayern,MULTIPOLYGON [3868 vertices #64040c6770]
2,Berlin,MULTIPOLYGON [238 vertices #26a120a249]
3,Brandenburg,MULTIPOLYGON [2479 vertices #8e93324db7]
4,Bremen,MULTIPOLYGON [156 vertices #e07b4369a5]
5,Hamburg,MULTIPOLYGON [244 vertices #abb1421ea1]
6,Hessen,MULTIPOLYGON [1621 vertices #493d8b85ad]
7,Mecklenburg-Vorpommern,MULTIPOLYGON [35026 vertices #7b284b605f]
8,Niedersachsen,MULTIPOLYGON [14321 vertices #46ceea567e]
9,Nordrhein-Westfalen,MULTIPOLYGON [2312 vertices #afd39a85c5]


answer 1 vs answer 2
   only in answer 1 (1): Mecklenburg-Vorpommern [MULTIPOLYGON [35026 vertices #7b284b605f]]
   only in answer 2 (1): Mecklenburg-Vorpowell [None]


In [115]:
pipeline("t029")

expected category: GEOMETRY_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok                                                                      extract_diff  sql_steps  sql_attempts sql_errors  \
model            agent_name                                                                                                                                                                   
GPT-4o mini      Agent-001   GEOMETRY_LOOKUP          1.0         1.0                                                                                          1.0           1.0              
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                                                                                          1.0           1.0              
Claude Haiku 5.5 Agent-001   GEOMETRY_LOOKUP          1.0         1.0                                                                                          1.0           1.0              
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                                                                                          1.0           1.0              
Qwen3-8B         Agent-001   GEOMETRY_LOOKUP          1.0         1.0                                                                                          1.0           1.0              
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                                                                                          1.0           1.0              
Qwen3-8B + QLoRA Agent-001   GEOMETRY_LOOKUP          1.0         0.0  entities: expected [{"entity_name": "badenwurttemberg", "entity_type": "state"},        1.0           1.0              
                 Agent-002   GEOMETRY_LOOKUP          1.0         0.0  entities: expected [{"entity_name": "badenwurttemberg", "entity_type": "state"},        1.0           1.0              

                             kqml_turns            status  seconds  tokens  
model            agent_name                                                 
GPT-4o mini      Agent-001          1.0          complete     9.99  4532.6  
                 Agent-002          0.0          complete     6.18  3527.6  
Claude Haiku 5.5 Agent-001          1.0          complete     9.30  7857.0  
                 Agent-002          0.0          complete     6.27  6240.6  
Qwen3-8B         Agent-001          1.0          complete    20.18  4732.0  
                 Agent-002          0.0          complete    16.67  3726.0  
Qwen3-8B + QLoRA Agent-001          1.0  partial-complete    21.71  4731.0  
                 Agent-002          1.0  partial-complete    22.16  4709.0

In [116]:
kqml("t029")          # kqml("t029", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5)

3 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 15 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5)
ASK  Agent-1 → Agent-2  (id req-076)
   asks for shape:  state Bayern
   asks for shape:  state Berlin
   asks for shape:  state Hessen
TELL  Agent-2 → Agent-1  (id req-076)
   found shape:     state Bayern: MULTIPOLYGON with 3869 points
   found shape:     state Berlin: MULTIPOLYGON with 239 points
   found shape:     state Hessen: MULTIPOLYGON with 1622 points
   → status of this reply: complete
   peer LLM tokens per reply: 1000–1612

=== exchange 2 · 5 ask(s) · Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-511)
   asks for shape:  state Bayern
   asks for shape:  state Berlin
   asks for shape:  state Hessen
   asks for shape:  state Mecklenburg-Vorpowell
TELL  Agent-2 → Agent-1  (id req-511)
   shap

In [117]:
Q["t029"] = """
SELECT 'state' AS kind, state_name AS name, ST_GeometryType(geo_shape) AS type,
       NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(geo_shape) p), 0) AS vertices
FROM states WHERE state_name IN ('Baden-Württemberg', 'Bayern', 'Berlin', 'Brandenburg', 'Bremen', 'Hamburg', 'Hessen', 'Mecklenburg-Vorpommern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Saarland', 'Sachsen', 'Sachsen-Anhalt', 'Schleswig-Holstein', 'Thüringen')
ORDER BY 1, 2
"""
r = sql(Q["t029"], qid="t029")
compare("t029", r, "name")

19 row(s)  (db = which database returned the row)


,kind,name,type,vertices,db
0,state,Baden-Württemberg,ST_MultiPolygon,5091.0,"Agent-001, Agent-002"
1,state,Bayern,NaN,NaN,Agent-001
2,state,Berlin,NaN,NaN,Agent-001
3,state,Brandenburg,ST_MultiPolygon,2479.0,"Agent-001, Agent-002"
4,state,Bremen,ST_MultiPolygon,156.0,"Agent-001, Agent-002"
5,state,Hamburg,ST_MultiPolygon,244.0,"Agent-001, Agent-002"
6,state,Hessen,NaN,NaN,Agent-001
7,state,Mecklenburg-Vorpommern,ST_MultiPolygon,35026.0,"Agent-001, Agent-002"
8,state,Niedersachsen,ST_MultiPolygon,14321.0,"Agent-001, Agent-002"
9,state,Nordrhein-Westfalen,ST_MultiPolygon,2312.0,"Agent-001, Agent-002"


answer 1: MATCH – SQL 16, answer 16 distinct names (0 duplicates)
answer 2: DIFFERS – SQL 16, answer 16 distinct names (0 duplicates)
   in SQL, missing in answer (1): Mecklenburg-Vorpommern
   in answer, not in SQL (1): Mecklenburg-Vorpowell


In [118]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t029", {1: "y", 2: "p"}, note="""
Answer 2 (Qwen3-8B + QLoRA): extraction typo 'Mecklenburg-Vorpowell', so 15 of 16 states;
the 16th is reported as missing. The 15 shapes it returned are correct.
""")


saved t029: answer 1 = correct, answer 2 = partly   (32/113 answers reviewed)


## Q29 · t030 · GEOMETRY_LOOKUP

**Where is Görlitz?**

⚠ **2 different answers** – see `diff` below.

> GPT-4o mini rejected the question as UNRELATED in all 10 asks; the other models return Görlitz's point.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [119]:
show("t030")
answer("t030", 1)
diff("t030")

**Q29 · t030 · GEOMETRY_LOOKUP** — Where is Görlitz?

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,30,Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5...,complete,1 item(s): Görlitz,correct,
2,10,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5),rejected,"{""status"": ""rejected""}",correct,


answer 1: status complete, type GEOMETRY_LOOKUP


,item,detail
0,Görlitz,POINT 14.9872 51.1528


answer 1 vs answer 2
   answer 1:  1 item(s): Görlitz
   answer 2:  {"status": "rejected"}


In [120]:
pipeline("t030")

expected category: GEOMETRY_LOOKUP   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                  
GPT-4o mini      Agent-001         UNRELATED          0.0         NaN                     0.0           NaN                    0.0  rejected     2.93  1453.0
                 Agent-002         UNRELATED          0.0         NaN                     0.0           NaN                    0.0  rejected     3.06  1453.0
Claude Haiku 5.5 Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete     9.82  7266.2
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     5.70  5689.8
Qwen3-8B         Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete    12.17  4263.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     8.45  3267.0
Qwen3-8B + QLoRA Agent-001   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    1.0  complete    13.09  4268.0
                 Agent-002   GEOMETRY_LOOKUP          1.0         1.0                     1.0           1.0                    0.0  complete     9.51  3272.0

In [121]:
kqml("t030")          # kqml("t030", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 15 ask(s) · Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-516)
   asks for shape:  city Görlitz
TELL  Agent-2 → Agent-1  (id req-516)
   found shape:     city Görlitz: POINT(14.9872 51.1528)
   → status of this reply: complete
   peer LLM tokens per reply: 996–1585



In [122]:
Q["t030"] = """
SELECT 'city' AS kind, city_name AS name, ST_AsText(centroid) AS type, 1 AS vertices
FROM cities WHERE city_name IN ('Görlitz')
ORDER BY 1, 2
"""
r = sql(Q["t030"], qid="t030")
compare("t030", r, "name")

1 row(s)  (db = which database returned the row)


,kind,name,type,vertices,db
0,city,Görlitz,POINT(14.9872 51.1528),1,Agent-002


answer 1: MATCH – SQL 1, answer 1 distinct names (0 duplicates)
answer 2: DIFFERS – SQL 1, answer 0 distinct names (0 duplicates)
   in SQL, missing in answer (1): Görlitz


In [123]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t030", {1: "y", 2: "n"}, note="""
Görlitz exists only in agent_2_db: POINT(14.9872 51.1528); Agent-001 got it via KQML.
Answer 2: GPT-4o mini classified the question as UNRELATED (10/10 asks) - classification error.
""")


saved t030: answer 1 = correct, answer 2 = wrong   (34/113 answers reviewed)


## Q30 · t031 · SPATIAL_OPERATION

**Merge Sachsen and Thüringen into one shape**

All 40 asks give the same answer.

**SQL check:** Compare type and vertex count with the answer (union).

How the query works:
- `ST_Union` merges the shapes into one
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [124]:
show("t031")
answer("t031", 1)

**Q30 · t031 · SPATIAL_OPERATION** — Merge Sachsen and Thüringen into one shape

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,"{""result"": {""geometry_type"": ""POLYGON"", ""srid"": 4326, ""wkt"": ""POLYGON [3292 vertices #9e33b04123]""}, ""status"": ""comp...",correct,


All 40 asks give the same answer.
answer 1: status complete, type SPATIAL_OPERATION
{
 "result": {
  "geometry_type": "POLYGON",
  "srid": 4326,
  "wkt": "POLYGON [3292 vertices #9e33b04123]"
 }
}


In [125]:
pipeline("t031")

expected category: SPATIAL_OPERATION   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                    
GPT-4o mini      Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete     6.04  4739.0
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete     6.11  4739.0
Claude Haiku 5.5 Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete     5.90  7921.0
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete     5.70  7921.0
Qwen3-8B         Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete    11.98  4866.0
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete    11.80  4866.0
Qwen3-8B + QLoRA Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete    13.18  4861.0
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete    12.65  4861.0

In [126]:
kqml("t031")          # kqml("t031", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [127]:
Q["t031"] = """
-- union of Sachsen, Thüringen
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_Union(ARRAY[{state:Sachsen}, {state:Thüringen}]) AS g) x
"""
r = sql(Q["t031"], qid="t031")

1 row(s)  (db = which database returned the row)


,type,vertices,area_km2,db
0,ST_Polygon,3292,34674,"Agent-001, Agent-002"


In [129]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t031", {1: "y"}, note="""

""")


saved t031: answer 1 = correct   (35/113 answers reviewed)


## Q31 · t032 · SPATIAL_OPERATION

**Give me the union of Nordrhein-Westfalen, Hessen and Niedersachsen**

All 40 asks give the same answer.

**SQL check:** Compare type and vertex count with the answer (union).

How the query works:
- `ST_Union` merges the shapes into one
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [130]:
show("t032")
answer("t032", 1)

**Q31 · t032 · SPATIAL_OPERATION** — Give me the union of Nordrhein-Westfalen, Hessen and Niedersachsen

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,"{""result"": {""geometry_type"": ""MULTIPOLYGON"", ""srid"": 4326, ""wkt"": ""MULTIPOLYGON [16026 vertices #85a7777153]""}, ""sta...",correct,


All 40 asks give the same answer.
answer 1: status complete, type SPATIAL_OPERATION
{
 "result": {
  "geometry_type": "MULTIPOLYGON",
  "srid": 4326,
  "wkt": "MULTIPOLYGON [16026 vertices #85a7777153]"
 }
}


In [131]:
pipeline("t032")

expected category: SPATIAL_OPERATION   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts                                                                        sql_errors  \
model            agent_name                                                                                                                                                                       
GPT-4o mini      Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           3.0                                                                                     
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           3.0                                                                                     
Claude Haiku 5.5 Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           3.0                                                                                     
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           3.2  SqlWriter response had no 'sql': '{"error": "The request names three shapes (Nor   
Qwen3-8B         Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           3.0                                                                                     
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           3.0                                                                                     
Qwen3-8B + QLoRA Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           3.0                                                                                     
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           3.0                                                                                     

                             kqml_turns    status  seconds   tokens  
model            agent_name                                          
GPT-4o mini      Agent-001          1.0  complete    10.65   6783.2  
                 Agent-002          0.0  complete     7.70   5798.0  
Claude Haiku 5.5 Agent-001          1.0  complete    12.60  11528.6  
                 Agent-002          0.0  complete    10.59  10459.6  
Qwen3-8B         Agent-001          1.0  complete    17.95   6969.0  
                 Agent-002          0.0  complete    14.17   5977.0  
Qwen3-8B + QLoRA Agent-001          1.0  complete    19.55   6964.0  
                 Agent-002          0.0  complete    15.82   5972.0

In [132]:
kqml("t032")          # kqml("t032", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A2 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A2 (5)

1 different KQML exchange(s) (request ids and token counts ignored)

=== exchange 1 · 20 ask(s) · GPT-4o mini / A1 (5); Claude Haiku 5.5 / A1 (5); Qwen3-8B / A1 (5); Qwen3-8B + QLoRA / A1 (5)
ASK  Agent-1 → Agent-2  (id req-521)
   asks for shape:  state Hessen
TELL  Agent-2 → Agent-1  (id req-521)
   found shape:     state Hessen: MULTIPOLYGON with 1622 points
   → status of this reply: complete
   peer LLM tokens per reply: 985–1586



In [133]:
Q["t032"] = """
-- union of Nordrhein-Westfalen, Hessen, Niedersachsen
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_Union(ARRAY[{state:Nordrhein-Westfalen}, {state:Hessen}, {state:Niedersachsen}]) AS g) x
"""
r = sql(Q["t032"], qid="t032")

1 row(s)  (db = which database returned the row)


,type,vertices,area_km2,db
0,ST_MultiPolygon,16026,103025,"Agent-001, Agent-002"


In [134]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t032", {1: "y"}, note="""
Agent-001 lacks Hessen's shape and got it via KQML before the union.
""")


saved t032: answer 1 = correct   (36/113 answers reviewed)


## Q32 · t033 · SPATIAL_OPERATION

**Combine Rheinland-Pfalz and Saarland into a single region**

All 40 asks give the same answer.

**SQL check:** Compare type and vertex count with the answer (union).

How the query works:
- `ST_Union` merges the shapes into one
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [135]:
show("t033")
answer("t033", 1)

**Q32 · t033 · SPATIAL_OPERATION** — Combine Rheinland-Pfalz and Saarland into a single region

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,"{""result"": {""geometry_type"": ""POLYGON"", ""srid"": 4326, ""wkt"": ""POLYGON [1355 vertices #49ba0ea776]""}, ""status"": ""comp...",correct,


All 40 asks give the same answer.
answer 1: status complete, type SPATIAL_OPERATION
{
 "result": {
  "geometry_type": "POLYGON",
  "srid": 4326,
  "wkt": "POLYGON [1355 vertices #49ba0ea776]"
 }
}


In [136]:
pipeline("t033")

expected category: SPATIAL_OPERATION   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                    
GPT-4o mini      Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete     6.38  4756.6
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete     6.24  4756.8
Claude Haiku 5.5 Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete     6.16  7951.0
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete     5.89  7951.0
Qwen3-8B         Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete    12.07  4896.0
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete    12.03  4896.0
Qwen3-8B + QLoRA Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete    14.11  4891.0
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete    13.96  4891.0

In [137]:
kqml("t033")          # kqml("t033", raw=True) also prints the raw JSON

No KQML – answered from its own database: GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5); Qwen3-8B / A2 (5); Qwen3-8B + QLoRA / A1 (5); Qwen3-8B + QLoRA / A2 (5)

None of the 40 asks used KQML.


In [138]:
Q["t033"] = """
-- union of Rheinland-Pfalz, Saarland
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_Union(ARRAY[{state:Rheinland-Pfalz}, {state:Saarland}]) AS g) x
"""
r = sql(Q["t033"], qid="t033")

1 row(s)  (db = which database returned the row)


,type,vertices,area_km2,db
0,ST_Polygon,1355,22497,"Agent-001, Agent-002"


In [139]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t033", {1: "y"}, note="""

""")


saved t033: answer 1 = correct   (37/113 answers reviewed)


## Q33 · t034 · SPATIAL_OPERATION

**Create one area from Bayern and Baden-Württemberg**

All 40 asks give the same answer.

**SQL check:** Compare type and vertex count with the answer (union).

How the query works:
- `ST_Union` merges the shapes into one
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [140]:
show("t034")
answer("t034", 1)

**Q33 · t034 · SPATIAL_OPERATION** — Create one area from Bayern and Baden-Württemberg

,asks,given_by,status,answer,verdict_in_data,your_verdict
answer_no,,,,,,
1,40,GPT-4o mini / A1 (5); GPT-4o mini / A2 (5); Claude Haiku 5.5 / A1 (5); Claude Haiku 5.5 / A2 (5); Qwen3-8B / A1 (5);...,complete,"{""result"": {""geometry_type"": ""MULTIPOLYGON"", ""srid"": 4326, ""wkt"": ""MULTIPOLYGON [7153 vertices #19cff65e32]""}, ""stat...",correct,


All 40 asks give the same answer.
answer 1: status complete, type SPATIAL_OPERATION
{
 "result": {
  "geometry_type": "MULTIPOLYGON",
  "srid": 4326,
  "wkt": "MULTIPOLYGON [7153 vertices #19cff65e32]"
 }
}


In [141]:
pipeline("t034")

expected category: SPATIAL_OPERATION   (numbers are means over the 5 reps)


category  category_ok  extract_ok extract_diff  sql_steps  sql_attempts sql_errors  kqml_turns    status  seconds  tokens
model            agent_name                                                                                                                                    
GPT-4o mini      Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    1.0  complete     9.77  5723.4
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete     6.28  4738.0
Claude Haiku 5.5 Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    1.0  complete     9.08  9521.0
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete     6.00  7935.0
Qwen3-8B         Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    1.0  complete    15.39  5880.0
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete    12.31  4888.0
Qwen3-8B + QLoRA Agent-001   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    1.0  complete    17.21  5875.0
                 Agent-002   SPATIAL_OPERATION          1.0         1.0                     2.0           2.0                    0.0  complete    13.56  4883.0

In [ ]:
kqml("t034")          # kqml("t034", raw=True) also prints the raw JSON

In [ ]:
Q["t034"] = """
-- union of Bayern, Baden-Württemberg
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_Union(ARRAY[{state:Bayern}, {state:Baden-Württemberg}]) AS g) x
"""
r = sql(Q["t034"], qid="t034")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t034", {1: "y"}, note="""
Agent-001 got Bayern's shape via KQML.
""")


## Q34 · t035 · SPATIAL_OPERATION

**What area do Brandenburg and Berlin share?**

All 40 asks give the same answer.

**SQL check:** Compare type and vertex count with the answer (intersection).

How the query works:
- `ST_Intersection` keeps only the common part
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t035")
answer("t035", 1)

In [ ]:
pipeline("t035")

In [ ]:
kqml("t035")          # kqml("t035", raw=True) also prints the raw JSON

In [ ]:
Q["t035"] = """
-- intersection of Brandenburg, Berlin
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_Intersection({state:Brandenburg}, {state:Berlin}) AS g) x
"""
r = sql(Q["t035"], qid="t035")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t035", {1: "y"}, note="""
No shared area - the intersection is the common border line (same fingerprint as Berlin's
outline). The answer says so correctly.
""")


## Q35 · t036 · SPATIAL_OPERATION

**Intersect Hessen with Nordrhein-Westfalen**

All 40 asks give the same answer.

**SQL check:** Compare type and vertex count with the answer (intersection).

How the query works:
- `ST_Intersection` keeps only the common part
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t036")
answer("t036", 1)

In [ ]:
pipeline("t036")

In [ ]:
kqml("t036")          # kqml("t036", raw=True) also prints the raw JSON

In [ ]:
Q["t036"] = """
-- intersection of Hessen, Nordrhein-Westfalen
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_Intersection({state:Hessen}, {state:Nordrhein-Westfalen}) AS g) x
"""
r = sql(Q["t036"], qid="t036")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t036", {1: "y"}, note="""
No shared area - Hessen and NRW meet only along the border. The answer says so correctly.
""")


## Q36 · t037 · SPATIAL_OPERATION

**Give me Niedersachsen with Bremen removed**

All 40 asks give the same answer.

**SQL check:** Compare type and vertex count with the answer (difference).

How the query works:
- `ST_Difference(a, b)` = a with b cut out
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t037")
answer("t037", 1)

In [ ]:
pipeline("t037")

In [ ]:
kqml("t037")          # kqml("t037", raw=True) also prints the raw JSON

In [ ]:
Q["t037"] = """
-- difference of Niedersachsen, Bremen
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_Difference({state:Niedersachsen}, {state:Bremen}) AS g) x
"""
r = sql(Q["t037"], qid="t037")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t037", {1: "y"}, note="""

""")


## Q37 · t038 · SPATIAL_OPERATION

**Show Brandenburg minus Berlin**

All 40 asks give the same answer.

**SQL check:** Compare type and vertex count with the answer (difference).

How the query works:
- `ST_Difference(a, b)` = a with b cut out
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t038")
answer("t038", 1)

In [ ]:
pipeline("t038")

In [ ]:
kqml("t038")          # kqml("t038", raw=True) also prints the raw JSON

In [ ]:
Q["t038"] = """
-- difference of Brandenburg, Berlin
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_Difference({state:Brandenburg}, {state:Berlin}) AS g) x
"""
r = sql(Q["t038"], qid="t038")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t038", {1: "y"}, note="""
Result = Brandenburg's own shape (same fingerprint #8e93324db7): Berlin is not part of
Brandenburg's polygon, so nothing is cut out. Correct.
""")


## Q38 · t039 · SPATIAL_OPERATION

**Which areas belong to either Sachsen or Bayern but not both?**

All 40 asks give the same answer.

**SQL check:** Compare type and vertex count with the answer (symmetric difference).

How the query works:
- `ST_SymDifference` keeps the parts that lie in only one shape
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t039")
answer("t039", 1)

In [ ]:
pipeline("t039")

In [ ]:
kqml("t039")          # kqml("t039", raw=True) also prints the raw JSON

In [ ]:
Q["t039"] = """
-- symmetric difference of Sachsen, Bayern
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_SymDifference({state:Sachsen}, {state:Bayern}) AS g) x
"""
r = sql(Q["t039"], qid="t039")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t039", {1: "y"}, note="""
Sachsen and Bayern do not overlap, so the symmetric difference equals their union.
""")


## Q39 · t040 · SPATIAL_OPERATION

**Symmetric difference of Hessen and Thüringen**

All 40 asks give the same answer.

**SQL check:** Compare type and vertex count with the answer (symmetric difference).

How the query works:
- `ST_SymDifference` keeps the parts that lie in only one shape
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t040")
answer("t040", 1)

In [ ]:
pipeline("t040")

In [ ]:
kqml("t040")          # kqml("t040", raw=True) also prints the raw JSON

In [ ]:
Q["t040"] = """
-- symmetric difference of Hessen, Thüringen
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_SymDifference({state:Hessen}, {state:Thüringen}) AS g) x
"""
r = sql(Q["t040"], qid="t040")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t040", {1: "y"}, note="""
Hessen and Thüringen do not overlap, so the symmetric difference equals their union.
""")


## Q40 · t041 · SPATIAL_OPERATION

**Which of Hessen and Bayern lie within 80 km of Würzburg?**

All 40 asks give the same answer.

**SQL check:** A state counts if km ≤ 80; the answer marks it with meets_zone=1.

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t041")
answer("t041", 1)

In [ ]:
pipeline("t041")

In [ ]:
kqml("t041")          # kqml("t041", raw=True) also prints the raw JSON

In [ ]:
Q["t041"] = """
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {city:Würzburg}::geography) / 1000)::numeric, 1) AS km
FROM states WHERE state_name IN ('Hessen', 'Bayern')
ORDER BY km
"""
r = sql(Q["t041"], qid="t041")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t041", {1: "y"}, note="""
Bayern 0 km (Würzburg lies in Bayern), Hessen 44.3 km - both within 80 km.
""")


## Q41 · t042 · SPATIAL_OPERATION

**Which of Thüringen and Bayern lie within 40 km of Coburg?**

All 40 asks give the same answer.

**SQL check:** A state counts if km ≤ 40; the answer marks it with meets_zone=1.

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t042")
answer("t042", 1)

In [ ]:
pipeline("t042")

In [ ]:
kqml("t042")          # kqml("t042", raw=True) also prints the raw JSON

In [ ]:
Q["t042"] = """
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {city:Coburg}::geography) / 1000)::numeric, 1) AS km
FROM states WHERE state_name IN ('Thüringen', 'Bayern')
ORDER BY km
"""
r = sql(Q["t042"], qid="t042")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t042", {1: "y"}, note="""
Bayern 0 km, Thüringen 7.3 km. Agent-001 got Coburg and Bayern via KQML.
""")


## Q42 · t043 · SPATIAL_OPERATION

**Join Niedersachsen and Schleswig-Holstein into one area and tell me which cities are inside it**

All 40 asks give the same answer.

**SQL check:** Cities inside the union; compare with the answer's list.

How the query works:
- `ST_Within(point, shape)` keeps points inside the shape
- `ST_Union` merges the shapes into one
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t043")
answer("t043", 1)

In [ ]:
pipeline("t043")

In [ ]:
kqml("t043")          # kqml("t043", raw=True) also prints the raw JSON

In [ ]:
Q["t043"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_Within(centroid, ST_Union(ARRAY[{state:Niedersachsen}, {state:Schleswig-Holstein}]))
ORDER BY city_name
"""
r = sql(Q["t043"], qid="t043")
compare("t043", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t043", {1: "y"}, note="""
289 cities, same as the SQL. Agent-001 found 156 locally and got 133 from Agent-002
(spatial query with an exclude list).
""")


## Q43 · t044 · SPATIAL_OPERATION

**Combine Brandenburg and Mecklenburg-Vorpommern and list the cities inside the merged area**

All 40 asks give the same answer.

**SQL check:** Cities inside the union; compare with the answer's list.

How the query works:
- `ST_Within(point, shape)` keeps points inside the shape
- `ST_Union` merges the shapes into one
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t044")
answer("t044", 1)

In [ ]:
pipeline("t044")

In [ ]:
kqml("t044")          # kqml("t044", raw=True) also prints the raw JSON

In [ ]:
Q["t044"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_Within(centroid, ST_Union(ARRAY[{state:Brandenburg}, {state:Mecklenburg-Vorpommern}]))
ORDER BY city_name
"""
r = sql(Q["t044"], qid="t044")
compare("t044", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t044", {1: "y"}, note="""
100 cities, same as the SQL.
""")


## Q44 · t045 · SPATIAL_OPERATION

**Which cities remain in Brandenburg after Berlin is cut out?**

All 40 asks give the same answer.

**SQL check:** Cities inside the difference; compare with the answer's list.

How the query works:
- `ST_Within(point, shape)` keeps points inside the shape
- `ST_Difference(a, b)` = a with b cut out
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t045")
answer("t045", 1)

In [ ]:
pipeline("t045")

In [ ]:
kqml("t045")          # kqml("t045", raw=True) also prints the raw JSON

In [ ]:
Q["t045"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_Within(centroid, ST_Difference({state:Brandenburg}, {state:Berlin}))
ORDER BY city_name
"""
r = sql(Q["t045"], qid="t045")
compare("t045", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t045", {1: "y"}, note="""
75 cities, same as the SQL. Agent-001 needed 2 KQML turns: Berlin's shape, then the spatial query.
""")


## Q45 · t046 · SPATIAL_OPERATION

**Merge Schleswig-Holstein and Hamburg into one area and tell me the population of that area in 2019**

All 40 asks give the same answer.

**SQL check:** The first result is the shape, the second the values; the answer's total should be their sum.

How the query works:
- `ST_Union` merges the shapes into one
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `state_demographics JOIN states` = the yearly numbers of the named states
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it
- two statements: the notebook shows **result 1** and **result 2** separately

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t046")
answer("t046", 1)

In [ ]:
pipeline("t046")

In [ ]:
kqml("t046")          # kqml("t046", raw=True) also prints the raw JSON

In [ ]:
Q["t046"] = """
-- union of Schleswig-Holstein, Hamburg
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_Union(ARRAY[{state:Schleswig-Holstein}, {state:Hamburg}]) AS g) x;

SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Schleswig-Holstein', 'Hamburg')
  AND d.stat_year IN (2019)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t046"], qid="t046")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t046", {1: "y"}, note="""
Total 4,751,026 = Hamburg 1,847,253 + Schleswig-Holstein 2,903,773; each agent got the
missing value from the other via KQML.
""")


## Q46 · t047 · SPATIAL_OPERATION

**Join Sachsen and Sachsen-Anhalt into one shape and give me the total live births for 2016**

All 40 asks give the same answer.

**SQL check:** The first result is the shape, the second the values; the answer's total should be their sum.

How the query works:
- `ST_Union` merges the shapes into one
- `ST_GeometryType` = shape type; vertices = the shape's points (`ST_DumpPoints`) with x and y rounded to 5 decimals (≈ 1 m), each location counted once – exactly how the answer's `[N vertices]` is counted (`ST_NPoints` would also count each ring's repeated closing point and near-duplicate points from union/difference)
- `ST_Area(…::geography) / 1e6` = area in km²
- `state_demographics JOIN states` = the yearly numbers of the named states
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it
- two statements: the notebook shows **result 1** and **result 2** separately

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t047")
answer("t047", 1)

In [ ]:
pipeline("t047")

In [ ]:
kqml("t047")          # kqml("t047", raw=True) also prints the raw JSON

In [ ]:
Q["t047"] = """
-- union of Sachsen, Sachsen-Anhalt
SELECT ST_GeometryType(g) AS type, NULLIF((SELECT COUNT(DISTINCT (ROUND(ST_X((p).geom)::numeric, 5), ROUND(ST_Y((p).geom)::numeric, 5)))
               FROM ST_DumpPoints(g) p), 0) AS vertices,
       ROUND((ST_Area(g::geography) / 1e6)::numeric) AS area_km2
FROM (SELECT ST_Union(ARRAY[{state:Sachsen}, {state:Sachsen-Anhalt}]) AS g) x;

SELECT s.state_name, d.stat_year, d.live_births
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Sachsen', 'Sachsen-Anhalt')
  AND d.stat_year IN (2016)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t047"], qid="t047")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t047", {1: "y"}, note="""
Sachsen 2016 live births are missing in both databases, so partial-complete with total = null
is correct. Sachsen-Anhalt 9,335 matches.
""")


## Q47 · t048 · SPATIAL_RELATIONSHIP_BUFFER

**Which cities are within 25 km of Erfurt?**

All 40 asks give the same answer.

**SQL check:** Cities within 25 km of Erfurt.

How the query works:
- `ST_DWithin(…::geography, …, d)` keeps places closer than d metres
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t048")
answer("t048", 1)

In [ ]:
pipeline("t048")

In [ ]:
kqml("t048")          # kqml("t048", raw=True) also prints the raw JSON

In [ ]:
Q["t048"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_DWithin(centroid::geography, {city:Erfurt}::geography, 25000)
  AND city_name <> 'Erfurt'
ORDER BY city_name
"""
r = sql(Q["t048"], qid="t048")
compare("t048", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t048", {1: "y"}, note="""

""")


## Q48 · t050 · SPATIAL_RELATIONSHIP_BUFFER

**What cities lie within 10 km of Potsdam?**

All 40 asks give the same answer.

**SQL check:** Cities within 10 km of Potsdam.

How the query works:
- `ST_DWithin(…::geography, …, d)` keeps places closer than d metres
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t050")
answer("t050", 1)

In [ ]:
pipeline("t050")

In [ ]:
kqml("t050")          # kqml("t050", raw=True) also prints the raw JSON

In [ ]:
Q["t050"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_DWithin(centroid::geography, {city:Potsdam}::geography, 10000)
  AND city_name <> 'Potsdam'
ORDER BY city_name
"""
r = sql(Q["t050"], qid="t050")
compare("t050", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t050", {1: "y"}, note="""

""")


## Q49 · t051 · SPATIAL_RELATIONSHIP_BUFFER

**Which cities lie north of Baden-Württemberg?**

⚠ **3 different answers** – see `diff` below.

> Agent-002 returns 1195 cities, Claude/GPT on Agent-001 1190: missing are same-name cities (Essen 7.01 51.45, Münster, Coswig, Gronau, Petersberg). Likely cause: the KQML ask's `exclude` list holds city names only. Qwen on Agent-001 returns only 606: look at its KQML exchange – Agent-002 answered **HTTP 500 Internal Server Error**, so the peer's cities never arrived.

**SQL check:** Simple reading of 'north of Baden-Württemberg' (beyond its bounding box). Your system may use a different definition (e.g. a direction sector), so judge the differences, not only the count.

How the query works:
- `ST_YMax/YMin/XMax/XMin` = edges of the bounding box
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t051")
answer("t051", 1)
diff("t051")

In [ ]:
pipeline("t051")

In [ ]:
kqml("t051")          # kqml("t051", raw=True) also prints the raw JSON

In [ ]:
Q["t051"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_Y(centroid) > ST_YMax({state:Baden-Württemberg})
ORDER BY city_name
"""
r = sql(Q["t051"], qid="t051")
compare("t051", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t051", {1: "y", 2: "p", 3: "n"}, note="""
The system defines 'north' as a direction sector, not the bounding box of the check SQL,
so I compare the answers with each other.
Answer 1 (all models on Agent-002, 1195 cities): complete.
Answer 2 (GPT/Claude on Agent-001, 1190): misses Essen, Münster, Coswig, Gronau, Petersberg -
the KQML exclude list holds names only, so same-name cities in Agent-001 hid them.
Answer 3 (Qwen on Agent-001, 606): the peer answered HTTP 500 three times, half the cities are
missing, yet the status says complete.
""")


## Q50 · t052 · SPATIAL_RELATIONSHIP_BUFFER

**Which towns are near Kassel?**

All 40 asks give the same answer.

**SQL check:** 'Near' has no fixed distance: this lists cities up to 100 km with their distance. Check which radius the answer used and whether that is reasonable.

How the query works:
- `ST_DWithin(…::geography, …, d)` keeps places closer than d metres
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t052")
answer("t052", 1)

In [ ]:
pipeline("t052")

In [ ]:
kqml("t052")          # kqml("t052", raw=True) also prints the raw JSON

In [ ]:
Q["t052"] = """
SELECT city_name, ST_AsText(centroid) AS point,
       ROUND((ST_Distance(centroid::geography, {city:Kassel}::geography) / 1000)::numeric, 1) AS km
FROM cities
WHERE ST_DWithin(centroid::geography, {city:Kassel}::geography, 100000)
  AND city_name <> 'Kassel'
ORDER BY km
"""
r = sql(Q["t052"], qid="t052")
compare("t052", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t052", {1: "y"}, note="""
The system used 100 km for 'near'. 147 of the 148 cities within 100 km: Bad Salzuflen
(99.95 km) is lost because the circle is drawn as a 32-sided polygon. Acceptable.
""")


## Q51 · t053 · SPATIAL_RELATIONSHIP_BUFFER

**What cities are south of Niedersachsen?**

⚠ **3 different answers** – see `diff` below.

> Agent-002 returns 1075 cities, Claude/GPT on Agent-001 1069: missing are same-name cities (Brühl, Fürth, Neunkirchen, Sulzbach, Taufkirchen, Weingarten). Qwen on Agent-001 returns only 528: its KQML ask got **HTTP 500** back from Agent-002, so the peer's cities are missing.

**SQL check:** Simple reading of 'south of Niedersachsen' (beyond its bounding box). Your system may use a different definition (e.g. a direction sector), so judge the differences, not only the count.

How the query works:
- `ST_YMax/YMin/XMax/XMin` = edges of the bounding box
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t053")
answer("t053", 1)
diff("t053")

In [ ]:
pipeline("t053")

In [ ]:
kqml("t053")          # kqml("t053", raw=True) also prints the raw JSON

In [ ]:
Q["t053"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_Y(centroid) < ST_YMin({state:Niedersachsen})
ORDER BY city_name
"""
r = sql(Q["t053"], qid="t053")
compare("t053", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t053", {1: "y", 2: "n", 3: "p"}, note="""
Same pattern as Q49.
Answer 1 (Agent-002, 1075 cities): complete.
Answer 2 (Qwen on Agent-001, 528): the peer answered HTTP 500 three times; status still complete.
Answer 3 (GPT/Claude on Agent-001, 1069): misses Brühl, Fürth, Neunkirchen, Sulzbach, Taufkirchen,
Weingarten - same-name cities excluded by name in the KQML ask.
""")


## Q52 · t054 · SPATIAL_RELATIONSHIP_BUFFER

**List the cities east of Würzburg**

All 40 asks give the same answer.

**SQL check:** Simple reading of 'east of Würzburg' (beyond its bounding box). Your system may use a different definition (e.g. a direction sector), so judge the differences, not only the count.

How the query works:
- `ST_YMax/YMin/XMax/XMin` = edges of the bounding box
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t054")
answer("t054", 1)

In [ ]:
pipeline("t054")

In [ ]:
kqml("t054")          # kqml("t054", raw=True) also prints the raw JSON

In [ ]:
Q["t054"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_X(centroid) > ST_XMax({city:Würzburg})
ORDER BY city_name
"""
r = sql(Q["t054"], qid="t054")
compare("t054", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t054", {1: "y"}, note="""
East = direction sector 45-135° from Würzburg, not the bounding box of the check SQL.
All 40 asks agree.
""")


## Q53 · t055 · SPATIAL_RELATIONSHIP_BUFFER

**Cities within 15.5 km of Heidelberg**

All 40 asks give the same answer.

**SQL check:** Cities within 15.5 km of Heidelberg.

How the query works:
- `ST_DWithin(…::geography, …, d)` keeps places closer than d metres
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t055")
answer("t055", 1)

In [ ]:
pipeline("t055")

In [ ]:
kqml("t055")          # kqml("t055", raw=True) also prints the raw JSON

In [ ]:
Q["t055"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_DWithin(centroid::geography, {city:Heidelberg}::geography, 15500)
  AND city_name <> 'Heidelberg'
ORDER BY city_name
"""
r = sql(Q["t055"], qid="t055")
compare("t055", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t055", {1: "y"}, note="""

""")


## Q54 · t056 · SPATIAL_RELATIONSHIP_BUFFER

**Which cities are in Baden-Württemberg?**

⚠ **2 different answers** – see `diff` below.

> Agent-001 returns 280 cities, Agent-002 281. The extra one is Weingarten (8.53 49.05) – both Weingartens lie in Baden-Württemberg.

**SQL check:** Cities inside Baden-Württemberg.

How the query works:
- `ST_Within(point, shape)` keeps points inside the shape
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t056")
answer("t056", 1)
diff("t056")

In [ ]:
pipeline("t056")

In [ ]:
kqml("t056")          # kqml("t056", raw=True) also prints the raw JSON

In [ ]:
Q["t056"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_Within(centroid, {state:Baden-Württemberg})
ORDER BY city_name
"""
r = sql(Q["t056"], qid="t056")
compare("t056", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t056", {1: "p", 2: "y"}, note="""
Baden-Württemberg has two cities named Weingarten. Answer 2 (Agent-002, 281) has both.
Answer 1 (Agent-001, 280) lost Weingarten (8.53 49.05) because the KQML exclude list uses names only.
""")


## Q55 · t057 · SPATIAL_RELATIONSHIP_BUFFER

**List all the cities inside Thüringen**

All 40 asks give the same answer.

**SQL check:** Cities inside Thüringen.

How the query works:
- `ST_Within(point, shape)` keeps points inside the shape
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t057")
answer("t057", 1)

In [ ]:
pipeline("t057")

In [ ]:
kqml("t057")          # kqml("t057", raw=True) also prints the raw JSON

In [ ]:
Q["t057"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_Within(centroid, {state:Thüringen})
ORDER BY city_name
"""
r = sql(Q["t057"], qid="t057")
compare("t057", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t057", {1: "y"}, note="""

""")


## Q56 · t058 · SPATIAL_RELATIONSHIP_BUFFER

**What cities are located within Schleswig-Holstein?**

All 40 asks give the same answer.

**SQL check:** Cities inside Schleswig-Holstein.

How the query works:
- `ST_Within(point, shape)` keeps points inside the shape
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t058")
answer("t058", 1)

In [ ]:
pipeline("t058")

In [ ]:
kqml("t058")          # kqml("t058", raw=True) also prints the raw JSON

In [ ]:
Q["t058"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_Within(centroid, {state:Schleswig-Holstein})
ORDER BY city_name
"""
r = sql(Q["t058"], qid="t058")
compare("t058", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t058", {1: "y"}, note="""

""")


## Q57 · t059 · SPATIAL_RELATIONSHIP_BUFFER

**Which cities does Saarland contain?**

All 40 asks give the same answer.

**SQL check:** Cities inside Saarland.

How the query works:
- `ST_Within(point, shape)` keeps points inside the shape
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t059")
answer("t059", 1)

In [ ]:
pipeline("t059")

In [ ]:
kqml("t059")          # kqml("t059", raw=True) also prints the raw JSON

In [ ]:
Q["t059"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_Within(centroid, {state:Saarland})
ORDER BY city_name
"""
r = sql(Q["t059"], qid="t059")
compare("t059", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t059", {1: "y"}, note="""

""")


## Q58 · t060 · SPATIAL_RELATIONSHIP_BUFFER

**Which cities lie within 20 km of Hessen?**

All 40 asks give the same answer.

**SQL check:** Cities within 20 km of the state (inside it counts too).

How the query works:
- `ST_DWithin(…::geography, …, d)` keeps places closer than d metres
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t060")
answer("t060", 1)

In [ ]:
pipeline("t060")

In [ ]:
kqml("t060")          # kqml("t060", raw=True) also prints the raw JSON

In [ ]:
Q["t060"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_DWithin(centroid::geography, {state:Hessen}::geography, 20000)
ORDER BY city_name
"""
r = sql(Q["t060"], qid="t060")
compare("t060", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t060", {1: "y"}, note="""
274 items = 273 names + two different cities with the same name, both within 20 km.
Agent-001 first fetched Hessen's shape, then ran the spatial query.
""")


## Q59 · t061 · SPATIAL_RELATIONSHIP_BUFFER

**Which cities lie within 30 km of Saarland?**

All 40 asks give the same answer.

**SQL check:** Cities within 30 km of the state (inside it counts too).

How the query works:
- `ST_DWithin(…::geography, …, d)` keeps places closer than d metres
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t061")
answer("t061", 1)

In [ ]:
pipeline("t061")

In [ ]:
kqml("t061")          # kqml("t061", raw=True) also prints the raw JSON

In [ ]:
Q["t061"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_DWithin(centroid::geography, {state:Saarland}::geography, 30000)
ORDER BY city_name
"""
r = sql(Q["t061"], qid="t061")
compare("t061", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t061", {1: "y"}, note="""

""")


## Q60 · t062 · SPATIAL_ADJACENCY

**Which states border Niedersachsen?**

All 40 asks give the same answer.

**SQL check:** Neighbours of Niedersachsen.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t062")
answer("t062", 1)

In [ ]:
pipeline("t062")

In [ ]:
kqml("t062")          # kqml("t062", raw=True) also prints the raw JSON

In [ ]:
Q["t062"] = """
SELECT state_name FROM states
WHERE ST_DWithin(geo_shape, {state:Niedersachsen}, 0.001)
  AND state_name <> 'Niedersachsen'
ORDER BY state_name
"""
r = sql(Q["t062"], qid="t062")
compare("t062", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t062", {1: "y"}, note="""
All 9 neighbours are right, incl. the Bremen enclave. ST_Touches found only 5 because the
state borders overlap slightly; with a 100 m tolerance the check gives exactly these 9.
""")


## Q61 · t063 · SPATIAL_ADJACENCY

**What are the neighbouring states of Bayern?**

All 40 asks give the same answer.

**SQL check:** Neighbours of Bayern.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t063")
answer("t063", 1)

In [ ]:
pipeline("t063")

In [ ]:
kqml("t063")          # kqml("t063", raw=True) also prints the raw JSON

In [ ]:
Q["t063"] = """
SELECT state_name FROM states
WHERE ST_DWithin(geo_shape, {state:Bayern}, 0.001)
  AND state_name <> 'Bayern'
ORDER BY state_name
"""
r = sql(Q["t063"], qid="t063")
compare("t063", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t063", {1: "y"}, note="""

""")


## Q62 · t064 · SPATIAL_ADJACENCY

**Which federal states share a border with Mecklenburg-Vorpommern?**

All 40 asks give the same answer.

**SQL check:** Neighbours of Mecklenburg-Vorpommern.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t064")
answer("t064", 1)

In [ ]:
pipeline("t064")

In [ ]:
kqml("t064")          # kqml("t064", raw=True) also prints the raw JSON

In [ ]:
Q["t064"] = """
SELECT state_name FROM states
WHERE ST_DWithin(geo_shape, {state:Mecklenburg-Vorpommern}, 0.001)
  AND state_name <> 'Mecklenburg-Vorpommern'
ORDER BY state_name
"""
r = sql(Q["t064"], qid="t064")
compare("t064", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t064", {1: "y"}, note="""

""")


## Q63 · t065 · SPATIAL_ADJACENCY

**Does Hessen border Bayern?**

All 40 asks give the same answer.

**SQL check:** Yes/no question: compare with the answer's verdict.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `ST_Intersects` = the shapes touch or overlap
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t065")
answer("t065", 1)

In [ ]:
pipeline("t065")

In [ ]:
kqml("t065")          # kqml("t065", raw=True) also prints the raw JSON

In [ ]:
Q["t065"] = """
SELECT ST_DWithin({state:Hessen}, {state:Bayern}, 0.001) AS neighbours_100m,
       ST_Intersects({state:Hessen}, {state:Bayern}) AS intersects
"""
r = sql(Q["t065"], qid="t065")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t065", {1: "y"}, note="""

""")


## Q64 · t066 · SPATIAL_ADJACENCY

**Is Saarland adjacent to Baden-Württemberg?**

All 40 asks give the same answer.

**SQL check:** Yes/no question: compare with the answer's verdict.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `ST_Intersects` = the shapes touch or overlap
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t066")
answer("t066", 1)

In [ ]:
pipeline("t066")

In [ ]:
kqml("t066")          # kqml("t066", raw=True) also prints the raw JSON

In [ ]:
Q["t066"] = """
SELECT ST_DWithin({state:Saarland}, {state:Baden-Württemberg}, 0.001) AS neighbours_100m,
       ST_Intersects({state:Saarland}, {state:Baden-Württemberg}) AS intersects
"""
r = sql(Q["t066"], qid="t066")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t066", {1: "y"}, note="""
Saarland and Baden-Württemberg do not touch (Rheinland-Pfalz lies between) - no is correct.
""")


## Q65 · t067 · SPATIAL_ADJACENCY

**Which states touch Brandenburg?**

All 40 asks give the same answer.

**SQL check:** Neighbours of Brandenburg.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t067")
answer("t067", 1)

In [ ]:
pipeline("t067")

In [ ]:
kqml("t067")          # kqml("t067", raw=True) also prints the raw JSON

In [ ]:
Q["t067"] = """
SELECT state_name FROM states
WHERE ST_DWithin(geo_shape, {state:Brandenburg}, 0.001)
  AND state_name <> 'Brandenburg'
ORDER BY state_name
"""
r = sql(Q["t067"], qid="t067")
compare("t067", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t067", {1: "y"}, note="""

""")


## Q66 · t068 · SPATIAL_ADJACENCY

**Who are the neighbours of Rheinland-Pfalz?**

All 40 asks give the same answer.

**SQL check:** Neighbours of Rheinland-Pfalz.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t068")
answer("t068", 1)

In [ ]:
pipeline("t068")

In [ ]:
kqml("t068")          # kqml("t068", raw=True) also prints the raw JSON

In [ ]:
Q["t068"] = """
SELECT state_name FROM states
WHERE ST_DWithin(geo_shape, {state:Rheinland-Pfalz}, 0.001)
  AND state_name <> 'Rheinland-Pfalz'
ORDER BY state_name
"""
r = sql(Q["t068"], qid="t068")
compare("t068", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t068", {1: "y"}, note="""

""")


## Q67 · t069 · SPATIAL_ADJACENCY

**Which state borders both Sachsen and Bayern?**

⚠ **2 different answers** – see `diff` below.

> GPT-4o mini returns Thüringen twice (duplicate after merging local and peer results).

**SQL check:** States touching both.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t069")
answer("t069", 1)
diff("t069")

In [ ]:
pipeline("t069")

In [ ]:
kqml("t069")          # kqml("t069", raw=True) also prints the raw JSON

In [ ]:
Q["t069"] = """
SELECT state_name FROM states
WHERE ST_DWithin(geo_shape, {state:Sachsen}, 0.001)
  AND ST_DWithin(geo_shape, {state:Bayern}, 0.001)
  AND state_name NOT IN ('Sachsen', 'Bayern')
ORDER BY state_name
"""
r = sql(Q["t069"], qid="t069")
compare("t069", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t069", {1: "y", 2: "p"}, note="""
Thüringen is the only state bordering both. Answer 2 (GPT-4o mini) lists Thüringen twice:
right state, but the merge of local and peer results kept a duplicate.
""")


## Q68 · t070 · SPATIAL_ADJACENCY

**What was the population of the states bordering Hessen in 2018?**

All 40 asks give the same answer.

**SQL check:** The first result lists the neighbours, the second the values.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `state_demographics JOIN states` = the yearly numbers of the named states
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it
- two statements: the notebook shows **result 1** and **result 2** separately

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t070")
answer("t070", 1)

In [ ]:
pipeline("t070")

In [ ]:
kqml("t070")          # kqml("t070", raw=True) also prints the raw JSON

In [ ]:
Q["t070"] = """
SELECT state_name FROM states
WHERE ST_DWithin(geo_shape, {state:Hessen}, 0.001)
  AND state_name <> 'Hessen'
ORDER BY state_name;

-- then the values for those states:
SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Baden-Württemberg', 'Bayern', 'Niedersachsen', 'Nordrhein-Westfalen', 'Rheinland-Pfalz', 'Thüringen')
  AND d.stat_year IN (2018)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t070"], qid="t070")
compare("t070", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t070", {1: "y"}, note="""
Hessen's 6 neighbours with the right 2018 populations (NRW, Baden-Württemberg, Bayern via KQML
from Agent-002).
""")


## Q69 · t071 · SPATIAL_ADJACENCY

**Give me the marriages in 2017 of every state that borders Thüringen**

All 40 asks give the same answer.

**SQL check:** The first result lists the neighbours, the second the values.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `state_demographics JOIN states` = the yearly numbers of the named states
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it
- two statements: the notebook shows **result 1** and **result 2** separately

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t071")
answer("t071", 1)

In [ ]:
pipeline("t071")

In [ ]:
kqml("t071")          # kqml("t071", raw=True) also prints the raw JSON

In [ ]:
Q["t071"] = """
SELECT state_name FROM states
WHERE ST_DWithin(geo_shape, {state:Thüringen}, 0.001)
  AND state_name <> 'Thüringen'
ORDER BY state_name;

-- then the values for those states:
SELECT s.state_name, d.stat_year, d.marriages
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Bayern', 'Hessen', 'Niedersachsen', 'Sachsen', 'Sachsen-Anhalt')
  AND d.stat_year IN (2017)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t071"], qid="t071")
compare("t071", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t071", {1: "y"}, note="""
Thüringen's 5 neighbours. Sachsen 2017 marriages are missing in both databases, so
partial-complete is correct; the other 4 values match.
""")


## Q70 · t072 · SPATIAL_ADJACENCY

**Do Bremen and Hamburg share a border?**

All 40 asks give the same answer.

**SQL check:** Yes/no question: compare with the answer's verdict.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `ST_Intersects` = the shapes touch or overlap
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t072")
answer("t072", 1)

In [ ]:
pipeline("t072")

In [ ]:
kqml("t072")          # kqml("t072", raw=True) also prints the raw JSON

In [ ]:
Q["t072"] = """
SELECT ST_DWithin({state:Bremen}, {state:Hamburg}, 0.001) AS neighbours_100m,
       ST_Intersects({state:Bremen}, {state:Hamburg}) AS intersects
"""
r = sql(Q["t072"], qid="t072")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t072", {1: "y"}, note="""

""")


## Q71 · t073 · SPATIAL_ADJACENCY

**Which states border Hamburg?**

All 40 asks give the same answer.

**SQL check:** Neighbours of Hamburg.

How the query works:
- `ST_DWithin(a, b, 0.001)` = the shapes are less than ~100 m apart = neighbours (`ST_Touches` is too strict here: neighbouring state borders overlap slightly in this data)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t073")
answer("t073", 1)

In [ ]:
pipeline("t073")

In [ ]:
kqml("t073")          # kqml("t073", raw=True) also prints the raw JSON

In [ ]:
Q["t073"] = """
SELECT state_name FROM states
WHERE ST_DWithin(geo_shape, {state:Hamburg}, 0.001)
  AND state_name <> 'Hamburg'
ORDER BY state_name
"""
r = sql(Q["t073"], qid="t073")
compare("t073", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t073", {1: "y"}, note="""

""")


## Q72 · t074 · SPATIAL_DIRECTION

**Which states lie south of Hessen?**

All 40 asks give the same answer.

**SQL check:** Simple reading of 'south of Hessen': centroid latitude compared. Your system may use a stricter definition (e.g. the whole state beyond the border), so judge the differences rather than only the match.

How the query works:
- `ST_Centroid` = centre point; `ST_Y` = latitude (north–south), `ST_X` = longitude (east–west); `d_lat`/`d_lon` = difference to the reference
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t074")
answer("t074", 1)

In [ ]:
pipeline("t074")

In [ ]:
kqml("t074")          # kqml("t074", raw=True) also prints the raw JSON

In [ ]:
Q["t074"] = """
SELECT * FROM (
  SELECT state_name,
         ROUND((ST_Y(ST_Centroid(geo_shape)) - ST_Y(ST_Centroid({state:Hessen})))::numeric, 2) AS d_lat,
         ROUND((ST_X(ST_Centroid(geo_shape)) - ST_X(ST_Centroid({state:Hessen})))::numeric, 2) AS d_lon
  FROM states WHERE state_name <> 'Hessen'
) x
WHERE d_lat < 0
ORDER BY state_name
"""
r = sql(Q["t074"], qid="t074")
compare("t074", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t074", {1: "y"}, note="""
Direction = 90° sector from the reference state's centroid (south = 135-225°,
direction_sectors in the agent config). Bayern lies at ~125°, Rheinland-Pfalz and Saarland at
~240-247°, so only Baden-Württemberg counts. Consistent with that definition.
""")


## Q73 · t075 · SPATIAL_DIRECTION

**What states are east of Nordrhein-Westfalen?**

All 40 asks give the same answer.

**SQL check:** Simple reading of 'east of Nordrhein-Westfalen': centroid longitude compared. Your system may use a stricter definition (e.g. the whole state beyond the border), so judge the differences rather than only the match.

How the query works:
- `ST_Centroid` = centre point; `ST_Y` = latitude (north–south), `ST_X` = longitude (east–west); `d_lat`/`d_lon` = difference to the reference
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t075")
answer("t075", 1)

In [ ]:
pipeline("t075")

In [ ]:
kqml("t075")          # kqml("t075", raw=True) also prints the raw JSON

In [ ]:
Q["t075"] = """
SELECT * FROM (
  SELECT state_name,
         ROUND((ST_Y(ST_Centroid(geo_shape)) - ST_Y(ST_Centroid({state:Nordrhein-Westfalen})))::numeric, 2) AS d_lat,
         ROUND((ST_X(ST_Centroid(geo_shape)) - ST_X(ST_Centroid({state:Nordrhein-Westfalen})))::numeric, 2) AS d_lon
  FROM states WHERE state_name <> 'Nordrhein-Westfalen'
) x
WHERE d_lon > 0
ORDER BY state_name
"""
r = sql(Q["t075"], qid="t075")
compare("t075", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t075", {1: "y"}, note="""
Sector 45-135° from NRW's centroid: Bremen and Schleswig-Holstein (<45°) and
Baden-Württemberg (~153°) fall outside. Consistent.
""")


## Q74 · t076 · SPATIAL_DIRECTION

**Which federal states are west of Sachsen?**

All 40 asks give the same answer.

**SQL check:** Simple reading of 'west of Sachsen': centroid longitude compared. Your system may use a stricter definition (e.g. the whole state beyond the border), so judge the differences rather than only the match.

How the query works:
- `ST_Centroid` = centre point; `ST_Y` = latitude (north–south), `ST_X` = longitude (east–west); `d_lat`/`d_lon` = difference to the reference
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t076")
answer("t076", 1)

In [ ]:
pipeline("t076")

In [ ]:
kqml("t076")          # kqml("t076", raw=True) also prints the raw JSON

In [ ]:
Q["t076"] = """
SELECT * FROM (
  SELECT state_name,
         ROUND((ST_Y(ST_Centroid(geo_shape)) - ST_Y(ST_Centroid({state:Sachsen})))::numeric, 2) AS d_lat,
         ROUND((ST_X(ST_Centroid(geo_shape)) - ST_X(ST_Centroid({state:Sachsen})))::numeric, 2) AS d_lon
  FROM states WHERE state_name <> 'Sachsen'
) x
WHERE d_lon < 0
ORDER BY state_name
"""
r = sql(Q["t076"], qid="t076")
compare("t076", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t076", {1: "y"}, note="""
Sector 225-315°: Bayern (~223°) and Mecklenburg-Vorpommern (~343°) fall just outside. Consistent.
""")


## Q75 · t077 · SPATIAL_DIRECTION

**Is Bayern south of Thüringen?**

All 40 asks give the same answer.

**SQL check:** Compare centroid latitude (north/south) or longitude (east/west) with the answer's yes/no. Your system may use a stricter definition – note which one it uses.

How the query works:
- `ST_Centroid` = centre point; `ST_Y` = latitude (north–south), `ST_X` = longitude (east–west); `d_lat`/`d_lon` = difference to the reference
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t077")
answer("t077", 1)

In [ ]:
pipeline("t077")

In [ ]:
kqml("t077")          # kqml("t077", raw=True) also prints the raw JSON

In [ ]:
Q["t077"] = """
SELECT ROUND(ST_Y(ST_Centroid({state:Bayern}))::numeric, 3) AS bayern_lat,
       ROUND(ST_Y(ST_Centroid({state:Thüringen}))::numeric, 3) AS thürin_lat,
       ROUND(ST_X(ST_Centroid({state:Bayern}))::numeric, 3) AS bayern_lon,
       ROUND(ST_X(ST_Centroid({state:Thüringen}))::numeric, 3) AS thürin_lon
"""
r = sql(Q["t077"], qid="t077")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t077", {1: "y"}, note="""

""")


## Q76 · t078 · SPATIAL_DIRECTION

**Which states are north of Bayern?**

All 40 asks give the same answer.

**SQL check:** Simple reading of 'north of Bayern': centroid latitude compared. Your system may use a stricter definition (e.g. the whole state beyond the border), so judge the differences rather than only the match.

How the query works:
- `ST_Centroid` = centre point; `ST_Y` = latitude (north–south), `ST_X` = longitude (east–west); `d_lat`/`d_lon` = difference to the reference
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t078")
answer("t078", 1)

In [ ]:
pipeline("t078")

In [ ]:
kqml("t078")          # kqml("t078", raw=True) also prints the raw JSON

In [ ]:
Q["t078"] = """
SELECT * FROM (
  SELECT state_name,
         ROUND((ST_Y(ST_Centroid(geo_shape)) - ST_Y(ST_Centroid({state:Bayern})))::numeric, 2) AS d_lat,
         ROUND((ST_X(ST_Centroid(geo_shape)) - ST_X(ST_Centroid({state:Bayern})))::numeric, 2) AS d_lon
  FROM states WHERE state_name <> 'Bayern'
) x
WHERE d_lat > 0
ORDER BY state_name
"""
r = sql(Q["t078"], qid="t078")
compare("t078", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t078", {1: "y"}, note="""
Sector 315-45°: Hessen, NRW, Rheinland-Pfalz and Saarland lie north-west (~300-305°), outside. Consistent.
""")


## Q77 · t079 · SPATIAL_DIRECTION

**Is Hamburg north of Bremen?**

All 40 asks give the same answer.

> All models say Hamburg is **not** north of Bremen, although Hamburg lies further north by latitude. Check which definition the system uses.

**SQL check:** Compare centroid latitude (north/south) or longitude (east/west) with the answer's yes/no. Your system may use a stricter definition – note which one it uses.

How the query works:
- `ST_Centroid` = centre point; `ST_Y` = latitude (north–south), `ST_X` = longitude (east–west); `d_lat`/`d_lon` = difference to the reference
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t079")
answer("t079", 1)

In [ ]:
pipeline("t079")

In [ ]:
kqml("t079")          # kqml("t079", raw=True) also prints the raw JSON

In [ ]:
Q["t079"] = """
SELECT ROUND(ST_Y(ST_Centroid({state:Hamburg}))::numeric, 3) AS hambur_lat,
       ROUND(ST_Y(ST_Centroid({state:Bremen}))::numeric, 3) AS bremen_lat,
       ROUND(ST_X(ST_Centroid({state:Hamburg}))::numeric, 3) AS hambur_lon,
       ROUND(ST_X(ST_Centroid({state:Bremen}))::numeric, 3) AS bremen_lon
"""
r = sql(Q["t079"], qid="t079")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t079", {1: "y"}, note="""
Hamburg's centroid lies at a bearing of ~70° from Bremen's (east-north-east), outside the north
sector 315-45°. So 'no' is consistent with the system's definition, although Hamburg is further
north by latitude.
""")


## Q78 · t080 · SPATIAL_DIRECTION

**Which states lie east of Niedersachsen?**

All 40 asks give the same answer.

**SQL check:** Simple reading of 'east of Niedersachsen': centroid longitude compared. Your system may use a stricter definition (e.g. the whole state beyond the border), so judge the differences rather than only the match.

How the query works:
- `ST_Centroid` = centre point; `ST_Y` = latitude (north–south), `ST_X` = longitude (east–west); `d_lat`/`d_lon` = difference to the reference
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t080")
answer("t080", 1)

In [ ]:
pipeline("t080")

In [ ]:
kqml("t080")          # kqml("t080", raw=True) also prints the raw JSON

In [ ]:
Q["t080"] = """
SELECT * FROM (
  SELECT state_name,
         ROUND((ST_Y(ST_Centroid(geo_shape)) - ST_Y(ST_Centroid({state:Niedersachsen})))::numeric, 2) AS d_lat,
         ROUND((ST_X(ST_Centroid(geo_shape)) - ST_X(ST_Centroid({state:Niedersachsen})))::numeric, 2) AS d_lon
  FROM states WHERE state_name <> 'Niedersachsen'
) x
WHERE d_lon > 0
ORDER BY state_name
"""
r = sql(Q["t080"], qid="t080")
compare("t080", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t080", {1: "y"}, note="""
Sector 45-135°: Thüringen (~135°) is borderline, Schleswig-Holstein (~24°) and Bayern (~150°)
are outside. Consistent.
""")


## Q79 · t081 · SPATIAL_DIRECTION

**States to the west of Brandenburg**

All 40 asks give the same answer.

**SQL check:** Simple reading of 'west of Brandenburg': centroid longitude compared. Your system may use a stricter definition (e.g. the whole state beyond the border), so judge the differences rather than only the match.

How the query works:
- `ST_Centroid` = centre point; `ST_Y` = latitude (north–south), `ST_X` = longitude (east–west); `d_lat`/`d_lon` = difference to the reference
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t081")
answer("t081", 1)

In [ ]:
pipeline("t081")

In [ ]:
kqml("t081")          # kqml("t081", raw=True) also prints the raw JSON

In [ ]:
Q["t081"] = """
SELECT * FROM (
  SELECT state_name,
         ROUND((ST_Y(ST_Centroid(geo_shape)) - ST_Y(ST_Centroid({state:Brandenburg})))::numeric, 2) AS d_lat,
         ROUND((ST_X(ST_Centroid(geo_shape)) - ST_X(ST_Centroid({state:Brandenburg})))::numeric, 2) AS d_lon
  FROM states WHERE state_name <> 'Brandenburg'
) x
WHERE d_lon < 0
ORDER BY state_name
"""
r = sql(Q["t081"], qid="t081")
compare("t081", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t081", {1: "y"}, note="""
Sector 225-315°: Bayern (~209°), Sachsen (~182°) and Mecklenburg-Vorpommern (~326°) are outside. Consistent.
""")


## Q80 · t082 · SPATIAL_DIRECTION

**Is Saarland west of Hessen?**

All 40 asks give the same answer.

**SQL check:** Compare centroid latitude (north/south) or longitude (east/west) with the answer's yes/no. Your system may use a stricter definition – note which one it uses.

How the query works:
- `ST_Centroid` = centre point; `ST_Y` = latitude (north–south), `ST_X` = longitude (east–west); `d_lat`/`d_lon` = difference to the reference
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t082")
answer("t082", 1)

In [ ]:
pipeline("t082")

In [ ]:
kqml("t082")          # kqml("t082", raw=True) also prints the raw JSON

In [ ]:
Q["t082"] = """
SELECT ROUND(ST_Y(ST_Centroid({state:Saarland}))::numeric, 3) AS saarla_lat,
       ROUND(ST_Y(ST_Centroid({state:Hessen}))::numeric, 3) AS hessen_lat,
       ROUND(ST_X(ST_Centroid({state:Saarland}))::numeric, 3) AS saarla_lon,
       ROUND(ST_X(ST_Centroid({state:Hessen}))::numeric, 3) AS hessen_lon
"""
r = sql(Q["t082"], qid="t082")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t082", {1: "y"}, note="""

""")


## Q81 · t083 · SPATIAL_DIRECTION

**What was the population in 2019 of the states north of Hessen?**

All 40 asks give the same answer.

**SQL check:** Simple reading of 'north of Hessen': centroid latitude compared. Your system may use a stricter definition (e.g. the whole state beyond the border), so judge the differences rather than only the match. The second result has the values.

How the query works:
- `ST_Centroid` = centre point; `ST_Y` = latitude (north–south), `ST_X` = longitude (east–west); `d_lat`/`d_lon` = difference to the reference
- `state_demographics JOIN states` = the yearly numbers of the named states
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it
- two statements: the notebook shows **result 1** and **result 2** separately

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t083")
answer("t083", 1)

In [ ]:
pipeline("t083")

In [ ]:
kqml("t083")          # kqml("t083", raw=True) also prints the raw JSON

In [ ]:
Q["t083"] = """
SELECT * FROM (
  SELECT state_name,
         ROUND((ST_Y(ST_Centroid(geo_shape)) - ST_Y(ST_Centroid({state:Hessen})))::numeric, 2) AS d_lat,
         ROUND((ST_X(ST_Centroid(geo_shape)) - ST_X(ST_Centroid({state:Hessen})))::numeric, 2) AS d_lon
  FROM states WHERE state_name <> 'Hessen'
) x
WHERE d_lat > 0
ORDER BY state_name;

-- then the values for those states:
SELECT s.state_name, d.stat_year, d.population
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Bremen', 'Hamburg', 'Niedersachsen', 'Schleswig-Holstein')
  AND d.stat_year IN (2019)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t083"], qid="t083")
compare("t083", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t083", {1: "y"}, note="""
North sector from Hessen: Niedersachsen, Bremen, Hamburg, Schleswig-Holstein; the 2019 values
match both databases.
""")


## Q82 · t084 · SPATIAL_DIRECTION

**Which states are located south of Schleswig-Holstein?**

All 40 asks give the same answer.

**SQL check:** Simple reading of 'south of Schleswig-Holstein': centroid latitude compared. Your system may use a stricter definition (e.g. the whole state beyond the border), so judge the differences rather than only the match.

How the query works:
- `ST_Centroid` = centre point; `ST_Y` = latitude (north–south), `ST_X` = longitude (east–west); `d_lat`/`d_lon` = difference to the reference
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t084")
answer("t084", 1)

In [ ]:
pipeline("t084")

In [ ]:
kqml("t084")          # kqml("t084", raw=True) also prints the raw JSON

In [ ]:
Q["t084"] = """
SELECT * FROM (
  SELECT state_name,
         ROUND((ST_Y(ST_Centroid(geo_shape)) - ST_Y(ST_Centroid({state:Schleswig-Holstein})))::numeric, 2) AS d_lat,
         ROUND((ST_X(ST_Centroid(geo_shape)) - ST_X(ST_Centroid({state:Schleswig-Holstein})))::numeric, 2) AS d_lon
  FROM states WHERE state_name <> 'Schleswig-Holstein'
) x
WHERE d_lat < 0
ORDER BY state_name
"""
r = sql(Q["t084"], qid="t084")
compare("t084", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t084", {1: "y"}, note="""
Sector 135-225°: Berlin, Brandenburg, Mecklenburg-Vorpommern (east) and Sachsen (~131°) are outside. Consistent.
""")


## Q83 · t085 · SPATIAL_DISTANCE

**Which states are within 50 km of Göttingen?**

All 40 asks give the same answer.

**SQL check:** States with km ≤ 50 (0 = the place lies inside the state).

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t085")
answer("t085", 1)

In [ ]:
pipeline("t085")

In [ ]:
kqml("t085")          # kqml("t085", raw=True) also prints the raw JSON

In [ ]:
Q["t085"] = """
SELECT * FROM (
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {city:Göttingen}::geography) / 1000)::numeric, 1) AS km
FROM states
ORDER BY km
) x
WHERE km <= 50
ORDER BY km
"""
r = sql(Q["t085"], qid="t085")
compare("t085", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t085", {1: "y"}, note="""

""")


## Q84 · t086 · SPATIAL_DISTANCE

**Which federal states lie within 120 km of Nürnberg?**

All 40 asks give the same answer.

**SQL check:** States with km ≤ 120 (0 = the place lies inside the state).

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t086")
answer("t086", 1)

In [ ]:
pipeline("t086")

In [ ]:
kqml("t086")          # kqml("t086", raw=True) also prints the raw JSON

In [ ]:
Q["t086"] = """
-- the databases store Nürnberg as 'Nuremberg'
SELECT * FROM (
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {city:Nuremberg}::geography) / 1000)::numeric, 1) AS km
FROM states
ORDER BY km
) x
WHERE km <= 120
ORDER BY km
"""
r = sql(Q["t086"], qid="t086")
compare("t086", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t086", {1: "y"}, note="""
Nürnberg is stored as 'Nuremberg' and the agents resolved it. Bayern 0, Baden-Württemberg 66.0,
Thüringen 86.7, Sachsen 118.7 km - all within 120 km.
""")


## Q85 · t087 · SPATIAL_DISTANCE

**States within 30 km of Mannheim**

All 40 asks give the same answer.

**SQL check:** States with km ≤ 30 (0 = the place lies inside the state).

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t087")
answer("t087", 1)

In [ ]:
pipeline("t087")

In [ ]:
kqml("t087")          # kqml("t087", raw=True) also prints the raw JSON

In [ ]:
Q["t087"] = """
SELECT * FROM (
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {city:Mannheim}::geography) / 1000)::numeric, 1) AS km
FROM states
ORDER BY km
) x
WHERE km <= 30
ORDER BY km
"""
r = sql(Q["t087"], qid="t087")
compare("t087", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t087", {1: "y"}, note="""

""")


## Q86 · t088 · SPATIAL_DISTANCE

**Is Thüringen within 100 km of Hannover?**

All 40 asks give the same answer.

**SQL check:** All states with their distance; the asked state counts if km ≤ 100.

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t088")
answer("t088", 1)

In [ ]:
pipeline("t088")

In [ ]:
kqml("t088")          # kqml("t088", raw=True) also prints the raw JSON

In [ ]:
Q["t088"] = """
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {city:Hannover}::geography) / 1000)::numeric, 1) AS km
FROM states
ORDER BY km
"""
r = sql(Q["t088"], qid="t088")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t088", {1: "y"}, note="""
Thüringen is 98.2 km from Hannover - yes is correct.
""")


## Q87 · t089 · SPATIAL_DISTANCE

**Which states are within 200 km of Leipzig?**

All 40 asks give the same answer.

**SQL check:** States with km ≤ 200 (0 = the place lies inside the state).

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t089")
answer("t089", 1)

In [ ]:
pipeline("t089")

In [ ]:
kqml("t089")          # kqml("t089", raw=True) also prints the raw JSON

In [ ]:
Q["t089"] = """
SELECT * FROM (
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {city:Leipzig}::geography) / 1000)::numeric, 1) AS km
FROM states
ORDER BY km
) x
WHERE km <= 200
ORDER BY km
"""
r = sql(Q["t089"], qid="t089")
compare("t089", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t089", {1: "y"}, note="""

""")


## Q88 · t090 · SPATIAL_DISTANCE

**Which states lie within 75 km of Magdeburg?**

All 40 asks give the same answer.

**SQL check:** States with km ≤ 75 (0 = the place lies inside the state).

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t090")
answer("t090", 1)

In [ ]:
pipeline("t090")

In [ ]:
kqml("t090")          # kqml("t090", raw=True) also prints the raw JSON

In [ ]:
Q["t090"] = """
SELECT * FROM (
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {city:Magdeburg}::geography) / 1000)::numeric, 1) AS km
FROM states
ORDER BY km
) x
WHERE km <= 75
ORDER BY km
"""
r = sql(Q["t090"], qid="t090")
compare("t090", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t090", {1: "y"}, note="""

""")


## Q89 · t091 · SPATIAL_DISTANCE

**Is Bremen within 150 km of Hamburg?**

All 40 asks give the same answer.

**SQL check:** All states with their distance; the asked state counts if km ≤ 150.

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t091")
answer("t091", 1)

In [ ]:
pipeline("t091")

In [ ]:
kqml("t091")          # kqml("t091", raw=True) also prints the raw JSON

In [ ]:
Q["t091"] = """
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {state:Hamburg}::geography) / 1000)::numeric, 1) AS km
FROM states WHERE state_name <> 'Hamburg'
ORDER BY km
"""
r = sql(Q["t091"], qid="t091")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t091", {1: "y"}, note="""

""")


## Q90 · t092 · SPATIAL_DISTANCE

**Which states are within 40 km of Saarland?**

All 40 asks give the same answer.

**SQL check:** States with km ≤ 40 (0 = the place lies inside the state).

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t092")
answer("t092", 1)

In [ ]:
pipeline("t092")

In [ ]:
kqml("t092")          # kqml("t092", raw=True) also prints the raw JSON

In [ ]:
Q["t092"] = """
SELECT * FROM (
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {state:Saarland}::geography) / 1000)::numeric, 1) AS km
FROM states WHERE state_name <> 'Saarland'
ORDER BY km
) x
WHERE km <= 40
ORDER BY km
"""
r = sql(Q["t092"], qid="t092")
compare("t092", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t092", {1: "y"}, note="""

""")


## Q91 · t093 · SPATIAL_DISTANCE

**Which states lie within 60 km of Frankfurt am Main?**

All 40 asks give the same answer.

> All models agree on **0 states** within 60 km of Frankfurt am Main – Frankfurt lies in Hessen, so this looks wrong. Check carefully.

**SQL check:** States with km ≤ 60 (0 = the place lies inside the state).

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t093")
answer("t093", 1)

In [ ]:
pipeline("t093")

In [ ]:
kqml("t093")          # kqml("t093", raw=True) also prints the raw JSON

In [ ]:
Q["t093"] = """
-- the databases store Frankfurt am Main as 'Frankfurt'
SELECT * FROM (
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {city:Frankfurt}::geography) / 1000)::numeric, 1) AS km
FROM states
ORDER BY km
) x
WHERE km <= 60
ORDER BY km
"""
r = sql(Q["t093"], qid="t093")
compare("t093", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t093", {1: "n"}, note="""
Wrong. Frankfurt am Main is stored as 'Frankfurt' (POINT 8.6822 50.1106) and no agent resolved
the name - both agents answered 'shape not found'. Correct answer: Hessen 0 km, Bayern 23.3,
Rheinland-Pfalz 29.2, Baden-Württemberg 53.4 km. The data marks it correct; it is not.
""")


## Q92 · t094 · SPATIAL_DISTANCE

**Give me the live births in 2016 of the states within 100 km of Kassel**

⚠ **2 different answers** – see `diff` below.

> Qwen3-8B's SQL failed (placeholder `year` not bound): status error in all 10 asks, but the data says correct. Also check whether Bayern really lies within 100 km of Kassel.

**SQL check:** States with km ≤ 100 (0 = the place lies inside the state). The second result has the values.

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `state_demographics JOIN states` = the yearly numbers of the named states
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it
- two statements: the notebook shows **result 1** and **result 2** separately

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t094")
answer("t094", 1)
diff("t094")

In [ ]:
pipeline("t094")

In [ ]:
kqml("t094")          # kqml("t094", raw=True) also prints the raw JSON

In [ ]:
Q["t094"] = """
SELECT * FROM (
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {city:Kassel}::geography) / 1000)::numeric, 1) AS km
FROM states
ORDER BY km
) x
WHERE km <= 100
ORDER BY km;

-- then the values for those states:
SELECT s.state_name, d.stat_year, d.live_births
FROM state_demographics d JOIN states s USING (state_id)
WHERE s.state_name IN ('Bayern', 'Hessen', 'Niedersachsen', 'Nordrhein-Westfalen', 'Sachsen-Anhalt', 'Thüringen')
  AND d.stat_year IN (2016)
ORDER BY s.state_name, d.stat_year
"""
r = sql(Q["t094"], qid="t094")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t094", {1: "y", 2: "n"}, note="""
Answer 1: 6 states within 100 km of Kassel (Bayern 94.0 km). Bayern's 2016 live births are
missing in both databases, so partial-complete is right; the other 5 values match.
Answer 2: Qwen3-8B misclassified it as SPATIAL_OPERATION and its SQL left :year unbound -
status error in all 10 asks.
""")


## Q93 · t095 · SPATIAL_DISTANCE

**Which states are within 90 km of Rostock?**

All 40 asks give the same answer.

**SQL check:** States with km ≤ 90 (0 = the place lies inside the state).

How the query works:
- `ST_Distance(…::geography) / 1000` gives the distance in km (0 = inside)
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t095")
answer("t095", 1)

In [ ]:
pipeline("t095")

In [ ]:
kqml("t095")          # kqml("t095", raw=True) also prints the raw JSON

In [ ]:
Q["t095"] = """
SELECT * FROM (
SELECT state_name,
       ROUND((ST_Distance(geo_shape::geography, {city:Rostock}::geography) / 1000)::numeric, 1) AS km
FROM states
ORDER BY km
) x
WHERE km <= 90
ORDER BY km
"""
r = sql(Q["t095"], qid="t095")
compare("t095", r, "state_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t095", {1: "y"}, note="""

""")


## Q94 · t096 · UNRELATED

**What is the population of Amsterdam?**

All 40 asks give the same answer.

**SQL check:** Out of scope – no SQL. Expected behaviour: the question is rejected.

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t096")
answer("t096", 1)

In [ ]:
pipeline("t096")

In [ ]:
kqml("t096")          # kqml("t096", raw=True) also prints the raw JSON

In [ ]:
Q["t096"] = """

"""
r = sql(Q["t096"], qid="t096")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t096", {1: "y"}, note="""

""")


## Q95 · t097 · UNRELATED

**Which cities lie within 25 km of Basel?**

All 40 asks give the same answer.

**SQL check:** Basel is not in Germany: expect 0 rows, so a rejection or 0 matches is right.

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t097")
answer("t097", 1)

In [ ]:
pipeline("t097")

In [ ]:
kqml("t097")          # kqml("t097", raw=True) also prints the raw JSON

In [ ]:
Q["t097"] = """
SELECT city_name, ST_AsText(centroid) AS point FROM cities WHERE city_name = 'Basel'
"""
r = sql(Q["t097"], qid="t097")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t097", {1: "y"}, note="""
Basel is in neither database; the agents searched, asked the peer and returned not-found.
Not rejected as UNRELATED, but the outcome is right.
""")


## Q96 · t098 · UNRELATED

**How do I bake sourdough bread?**

All 40 asks give the same answer.

**SQL check:** Out of scope – no SQL. Expected behaviour: the question is rejected.

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t098")
answer("t098", 1)

In [ ]:
pipeline("t098")

In [ ]:
kqml("t098")          # kqml("t098", raw=True) also prints the raw JSON

In [ ]:
Q["t098"] = """

"""
r = sql(Q["t098"], qid="t098")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t098", {1: "y"}, note="""

""")


## Q97 · t099 · UNRELATED

**Who won the 2014 football World Cup?**

All 40 asks give the same answer.

**SQL check:** Out of scope – no SQL. Expected behaviour: the question is rejected.

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t099")
answer("t099", 1)

In [ ]:
pipeline("t099")

In [ ]:
kqml("t099")          # kqml("t099", raw=True) also prints the raw JSON

In [ ]:
Q["t099"] = """

"""
r = sql(Q["t099"], qid="t099")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t099", {1: "y"}, note="""

""")


## Q98 · t100 · UNRELATED

**Which states border Texas?**

⚠ **2 different answers** – see `diff` below.

> Claude and most GPT asks reject it (UNRELATED); Qwen and 2 GPT asks run an adjacency query and return 0 matches.

**SQL check:** Out of scope – no SQL. Decide whether a rejection or '0 matches' counts as correct.

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("t100")
answer("t100", 1)
diff("t100")

In [ ]:
pipeline("t100")

In [ ]:
kqml("t100")          # kqml("t100", raw=True) also prints the raw JSON

In [ ]:
Q["t100"] = """

"""
r = sql(Q["t100"], qid="t100")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("t100", {1: "y", 2: "y"}, note="""
Texas is not a German state. Answer 1 (Qwen, 2 GPT asks) ran an adjacency query, found nothing
and flagged Texas as unknown; answer 2 (Claude, most GPT asks) rejected it. Both are acceptable.
""")


## Q99 · q-ee68a1a070 · (no category)

**Give me the point for Hamburg, the city**

All 40 asks give the same answer.

**SQL check:** Check that every place is found and the vertex count / point matches the answer.

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("q-ee68a1a070")
answer("q-ee68a1a070", 1)

In [ ]:
pipeline("q-ee68a1a070")

In [ ]:
kqml("q-ee68a1a070")          # kqml("q-ee68a1a070", raw=True) also prints the raw JSON

In [ ]:
Q["q-ee68a1a070"] = """
SELECT 'city' AS kind, city_name AS name, ST_AsText(centroid) AS type, 1 AS vertices
FROM cities WHERE city_name IN ('Hamburg')
ORDER BY 1, 2
"""
r = sql(Q["q-ee68a1a070"], qid="q-ee68a1a070")
compare("q-ee68a1a070", r, "name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("q-ee68a1a070", {1: "y"}, note="""
Agent-001 has Hamburg (city) without a point; it got POINT(10 53.55) via KQML.
""")


## Q100 · q-f21a844357 · (no category)

**Show me cities within 60 km of Hannover?**

All 40 asks give the same answer.

**SQL check:** Cities within 60 km of Hannover.

How the query works:
- `ST_DWithin(…::geography, …, d)` keeps places closer than d metres
- `{state:Name}` / `{city:Name}` = that shape, fetched from whichever database has it

Steps: answers → pipeline → KQML messages → SQL check → your review.

In [ ]:
show("q-f21a844357")
answer("q-f21a844357", 1)

In [ ]:
pipeline("q-f21a844357")

In [ ]:
kqml("q-f21a844357")          # kqml("q-f21a844357", raw=True) also prints the raw JSON

In [ ]:
Q["q-f21a844357"] = """
SELECT city_name, ST_AsText(centroid) AS point
FROM cities
WHERE ST_DWithin(centroid::geography, {city:Hannover}::geography, 60000)
  AND city_name <> 'Hannover'
ORDER BY city_name
"""
r = sql(Q["q-f21a844357"], qid="q-f21a844357")
compare("q-f21a844357", r, "city_name")

In [ ]:
# your review: y = correct, n = wrong, p = partly – per answer number from show(); then run
review("q-f21a844357", {1: "y"}, note="""

""")


## Results – progress and new answer accuracy

Copies each verdict to all asks with that answer, writes `review/answer_verdicts_per_ask.csv` and recomputes
answer accuracy per model on the reviewed asks.

In [ ]:
verdict_of = {k: r["verdict"] for k, r in reviews.items()}
per_ask = df[["model", "qid", "agent_name", "rep", "fingerprint", "answer_status", "answer_verdict"]].copy()
per_ask = per_ask.rename(columns={"answer_verdict": "verdict_before"})
per_ask["verdict_manual"] = [verdict_of.get((q, fp), "") for q, fp in zip(per_ask["qid"], per_ask["fingerprint"])]
per_ask.to_csv(PER_ASK_FILE, index=False, encoding="utf-8-sig")

reviewed = per_ask[per_ask["verdict_manual"] != ""]
print(f"asks with your verdict: {len(reviewed)} / {len(per_ask)}")
if len(reviewed):
    acc = reviewed.groupby("model", observed=True).agg(
        asks_reviewed=("qid", "size"),
        correct=("verdict_manual", lambda s: (s == "correct").mean()),
        partly=("verdict_manual", lambda s: (s == "partly").mean()),
        wrong=("verdict_manual", lambda s: (s == "wrong").mean()),
        accuracy_before=("verdict_before", lambda s: (s == "correct").mean()),
    )
    display(acc.rename(index=MODEL_LABEL).round(3))

In [ ]:
# Asks whose verdict changed compared with the data – these change the thesis numbers.
changed = reviewed[reviewed["verdict_before"].fillna("unreviewed") != reviewed["verdict_manual"]]
print(f"{len(changed)} asks changed")
(changed.groupby(["qid", "model", "verdict_before", "verdict_manual"], observed=True, dropna=False)
        .size().rename("asks").reset_index())

In [ ]:
# All your reviews with notes.
pd.read_csv(REVIEW_FILE, encoding="utf-8-sig") if REVIEW_FILE.exists() else "No reviews yet."